# Paper figures

Run the setup cell first. Review model, SAE, data paths, and run settings before starting. See the repository README for setup and pipeline order.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
# ============================================================
# Full replacement function (Sorted & High Contrast)
# ============================================================

DEFAULT_EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

def plot_all_model_ablation_summary(
    models,
    base_data_dir="data",
    output_dir="paper_visuals",
    skip_models=None,
    emotions=DEFAULT_EMOTIONS,
    show=True,
    save=True,
    dpi=300,
):
    """
    Plot only one figure:
    Max layer-wise ablation flip-away-from-true rate across models.

    Saves only:
        paper_visuals/heatmap_max_layerwise_ablation_flip_away_rate.png

    Does not save CSVs.
    Does not save run logs.
    Does not create other plots.
    """
    import os
    import re
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    import seaborn as sns

    if skip_models is None:
        skip_models = set()

    os.makedirs(output_dir, exist_ok=True)

    def get_sae_sizes_for_model(model_name):
        if model_name.startswith("google/gemma-2"):
            return [None]

        if model_name.startswith("meta-llama/"):
            return [None]

        if model_name.startswith("google/gemma-3"):
            return ["small", "big"]

        return [None]

    def get_model_slug(model_name):
        return model_name.replace("/", "-")

    def get_model_label(model_name, sae_size):
        slug = get_model_slug(model_name)

        if sae_size is None:
            return slug

        return f"{slug}-{sae_size}"

    def get_feature_dir(model_name, sae_size):
        slug = get_model_slug(model_name)

        if sae_size is None:
            return os.path.join(base_data_dir, f"{slug}-feature-act")

        return os.path.join(base_data_dir, f"{slug}-feature-act-{sae_size}")

    def prettify_model_label(label):
        label = label.replace("google-", "")
        label = label.replace("meta-llama-", "")
        label = label.replace("Llama-3.1-8B", "Llama-3.1-8B")
        label = label.replace("gemma-2-", "Gemma-2-")
        label = label.replace("gemma-3-", "Gemma-3-")
        label = label.replace("-feature-act", "")
        label = label.replace("-small", " (small)")
        label = label.replace("-big", " (big)")
        return label

    # Custom sorting rule: sorts by size (numerical), forces Llama models to the very bottom
    def make_sorting_key(prettified_label):
        is_llama = 1 if "Llama" in prettified_label else 0
        
        # Look for size pattern like 2B, 8B, 27B, 70B, etc.
        match = re.search(r"(\d+(?:\.\d+)?)[Bb]", prettified_label)
        size = float(match.group(1)) if match else 0.0
        
        # Tie-breaker for sub-variants (e.g., big vs small SAE variants)
        is_big_sae = 1 if "(big)" in prettified_label else 0
        
        return (is_llama, size, is_big_sae)

    all_dfs = []
    run_log = []

    for model_name in models:
        if model_name in skip_models:
            run_log.append(
                {
                    "model_name": model_name,
                    "sae_size": "none",
                    "status": "skipped",
                    "summary_path": "",
                }
            )
            continue

        for sae_size in get_sae_sizes_for_model(model_name):
            model_label = get_model_label(model_name, sae_size)
            feature_dir = get_feature_dir(model_name, sae_size)

            summary_path = os.path.join(
                feature_dir,
                "per_layer_per_emotion_feature_ablation",
                "per_layer_per_emotion_ablation_summary.csv",
            )

            if not os.path.exists(summary_path):
                print("Missing:", summary_path)
                run_log.append(
                    {
                        "model_name": model_name,
                        "sae_size": sae_size if sae_size is not None else "none",
                        "model_label": model_label,
                        "status": "missing",
                        "summary_path": summary_path,
                    }
                )
                continue

            df = pd.read_csv(summary_path)

            df["model_name"] = model_name
            df["sae_size"] = sae_size if sae_size is not None else "none"
            df["model_label"] = model_label

            all_dfs.append(df)

            print("Loaded:", summary_path)

            run_log.append(
                {
                    "model_name": model_name,
                    "sae_size": sae_size if sae_size is not None else "none",
                    "model_label": model_label,
                    "status": "loaded",
                    "summary_path": summary_path,
                }
            )

    run_log_df = pd.DataFrame(run_log)

    if len(all_dfs) == 0:
        print("No ablation summaries found.")

        return {
            "df": pd.DataFrame(),
            "best_df": pd.DataFrame(),
            "pivot": pd.DataFrame(),
            "figure": None,
            "run_log": run_log_df,
            "figures": {},
        }

    df = pd.concat(all_dfs, ignore_index=True)

    required_cols = [
        "model_label",
        "target_emotion",
        "layer_idx",
        "flip_away_from_true_rate",
    ]

    missing_cols = [col for col in required_cols if col not in df.columns]

    if len(missing_cols) > 0:
        raise ValueError(
            f"Missing required columns: {missing_cols}. "
            f"Available columns: {list(df.columns)}"
        )

    df["layer_idx"] = pd.to_numeric(df["layer_idx"], errors="coerce")
    df["flip_away_from_true_rate"] = pd.to_numeric(
        df["flip_away_from_true_rate"],
        errors="coerce",
    )

    df = df.dropna(
        subset=[
            "layer_idx",
            "flip_away_from_true_rate",
        ]
    ).copy()

    df["layer_idx"] = df["layer_idx"].astype(int)

    metric = "flip_away_from_true_rate"

    # Keep the layer with the maximum flip-away rate for each model and emotion.
    best_df = (
        df.sort_values(metric, ascending=False)
        .groupby(["model_label", "target_emotion"], as_index=False)
        .first()
        .sort_values(["model_label", "target_emotion"])
        .reset_index(drop=True)
    )

    pivot = best_df.pivot_table(
        index="model_label",
        columns="target_emotion",
        values=metric,
        aggfunc="mean",
    )

    pivot = pivot.reindex(
        columns=[emotion for emotion in emotions if emotion in pivot.columns]
    )

    if pivot.empty:
        print("Pivot table is empty. No figure generated.")

        return {
            "df": df,
            "best_df": best_df,
            "pivot": pivot,
            "figure": None,
            "run_log": run_log_df,
            "figures": {},
        }

    # Clean up labels for display
    display_pivot = pivot.copy()
    display_pivot.index = [prettify_model_label(x) for x in display_pivot.index]
    display_pivot.columns = [str(x).capitalize() for x in display_pivot.columns]

    # --- MODEL REORDERING PROTOCOL ---
    # Sort the index list based on the custom size & family rules
    sorted_index = sorted(display_pivot.index, key=make_sorting_key)
    display_pivot = display_pivot.reindex(sorted_index)

    # Set up paper typography (Clean sans-serif)
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    # Auto-adjust layout frames
    fig_height = max(4.0, 0.45 * len(display_pivot) + 1.0)
    fig_width = max(8.0, 0.95 * len(display_pivot.columns) + 2.5)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    # --- HIGH CONTRAST CALCULATION ---
    raw_values = display_pivot.values
    valid_values = raw_values[~np.isnan(raw_values)]
    
    if len(valid_values) > 0:
        data_min = np.min(valid_values)
        data_max = np.max(valid_values)
        # 5% soft color space buffer padding
        v_min_calibrated = max(0.0, data_min - 0.05)
        v_max_calibrated = min(1.0, data_max + 0.05)
    else:
        v_min_calibrated = 0.0
        v_max_calibrated = 1.0

    # Render optimized seaborn layout
    sns.heatmap(
        display_pivot,
        ax=ax,
        cmap="Blues_r",  # Dark for smaller values
        annot=True,
        fmt=".0%",
        linewidths=1.5,
        vmin=v_min_calibrated,
        vmax=v_max_calibrated,
        cbar_kws={
            "label": "Max Flip-Away Rate", 
            "fraction": 0.03, 
            "pad": 0.04
        },
        annot_kws={"fontsize": 9.5}
    )

    # Clean colorbar formatting using matplotlib's colorbar object hooked by seaborn
    cbar = ax.collections[0].colorbar
    cbar.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f"{x:.0%}"))
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(size=0)
    cbar.set_label("Max Flip-Away Rate", labelpad=12, fontsize=10, fontweight="medium")

    # Rotate column text slightly to match presentation standards
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    ax.set_yticklabels(ax.get_yticklabels(), fontweight="medium", rotation=0)

    # Strip messy axis tags
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(which="both", bottom=False, left=False)

    fig.tight_layout()

    figure_path = os.path.join(
        output_dir,
        "heatmap_max_layerwise_ablation_flip_away_rate.png",
    )

    if save:
        fig.savefig(
            figure_path,
            dpi=dpi,
            bbox_inches="tight",
            facecolor="white",
        )
        print("Saved figure:", figure_path)

    if show:
        plt.show()
    else:
        plt.close(fig)

    figures = {
        "heatmap_max_layerwise_ablation_flip_away_rate": figure_path if save else None
    }

    return {
        "df": df,
        "best_df": best_df,
        "pivot": pivot,
        "figure": figure_path if save else None,
        "run_log": run_log_df,
        "figures": figures,
    }

In [ ]:
# ============================================================
# Run code
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

results = plot_all_model_ablation_summary(
    models=MODELS,
    base_data_dir="data",
    output_dir="paper_visuals",
    skip_models={
        # Add model names here if needed.
        # Example:
        # "google/gemma-3-12b-pt",
    },
    emotions=DEFAULT_EMOTIONS,
    show=True,
    save=True,
    dpi=300,
)

ablation_all = results["df"]
best_ablation = results["best_df"]
pivot_ablation = results["pivot"]
figure_path = results["figure"]

# Optional compatibility with your previous variables.
run_log = results["run_log"]
figures = results["figures"]

print("Figure:", figure_path)

In [ ]:
# ============================================================
# Full replacement function (Log Scaling for Tight Multipliers)
# ============================================================

def run_all_model_publishable_feature_aggregate_plot(
    models,
    base_data_dir="data",
    output_dir="paper_visuals",
    skip_models=None,
    emotions=DEFAULT_EMOTIONS,
    show=True,
    save=True,
    dpi=300,
):
    """
    Plot only:
        Late-layer emotion feature concentration across models.

    Metric:
        mean_features_for_80pct_late_layers

    Interpretation:
        Lower value = more concentrated emotion feature representation.

    Saves only:
        paper_visuals/heatmap_late_layer_emotion_feature_concentration.png
    """
    import os
    import re
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    import seaborn as sns

    if skip_models is None:
        skip_models = set()

    os.makedirs(output_dir, exist_ok=True)

    def get_sae_sizes_for_model(model_name):
        if model_name.startswith("google/gemma-2"):
            return [None]

        if model_name.startswith("meta-llama/"):
            return [None]

        if model_name.startswith("google/gemma-3"):
            return ["small", "big"]

        return [None]

    def get_model_slug(model_name):
        return model_name.replace("/", "-")

    def get_model_label(model_name, sae_size):
        slug = get_model_slug(model_name)

        if sae_size is None:
            return slug

        return f"{slug}-{sae_size}"

    def get_feature_dir(model_name, sae_size):
        slug = get_model_slug(model_name)

        if sae_size is None:
            return os.path.join(base_data_dir, f"{slug}-feature-act")

        return os.path.join(base_data_dir, f"{slug}-feature-act-{sae_size}")

    def prettify_model_label(label):
        label = label.replace("google-", "")
        label = label.replace("meta-llama-", "")
        label = label.replace("Llama-3.1-8B", "Llama-3.1-8B")
        label = label.replace("gemma-2-", "Gemma-2-")
        label = label.replace("gemma-3-", "Gemma-3-")
        label = label.replace("-feature-act", "")
        label = label.replace("-small", " (small)")
        label = label.replace("-big", " (big)")
        return label

    # Custom sorting rule: sorts by size (numerical), forces Llama models to the very bottom
    def make_sorting_key(prettified_label):
        is_llama = 1 if "Llama" in prettified_label else 0
        
        # Look for size pattern like 2B, 8B, 27B, 70B, etc.
        match = re.search(r"(\d+(?:\.\d+)?)[Bb]", prettified_label)
        size = float(match.group(1)) if match else 0.0
        
        # Tie-breaker for sub-variants (e.g., big vs small SAE variants)
        is_big_sae = 1 if "(big)" in prettified_label else 0
        
        return (is_llama, size, is_big_sae)

    paper_summary_dfs = []
    run_log = []

    for model_name in models:
        if model_name in skip_models:
            run_log.append(
                {
                    "model_name": model_name,
                    "sae_size": "none",
                    "model_label": get_model_label(model_name, None),
                    "status": "skipped",
                    "path": "",
                }
            )
            continue

        for sae_size in get_sae_sizes_for_model(model_name):
            model_label = get_model_label(model_name, sae_size)
            feature_dir = get_feature_dir(model_name, sae_size)
            analysis_dir = os.path.join(feature_dir, "emotion_feature_analysis")

            paper_summary_path = os.path.join(
                analysis_dir,
                "paper_summary_table.csv",
            )

            if not os.path.exists(paper_summary_path):
                print("Missing:", paper_summary_path)
                run_log.append(
                    {
                        "model_name": model_name,
                        "sae_size": sae_size if sae_size is not None else "none",
                        "model_label": model_label,
                        "status": "missing",
                        "path": paper_summary_path,
                    }
                )
                continue

            df = pd.read_csv(paper_summary_path)

            df["model_name"] = model_name
            df["sae_size"] = sae_size if sae_size is not None else "none"
            df["model_label"] = model_label

            paper_summary_dfs.append(df)

            print("Loaded:", paper_summary_path)

            run_log.append(
                {
                    "model_name": model_name,
                    "sae_size": sae_size if sae_size is not None else "none",
                    "model_label": model_label,
                    "status": "loaded",
                    "path": paper_summary_path,
                }
            )

    run_log_df = pd.DataFrame(run_log)

    if len(paper_summary_dfs) == 0:
        print("No paper_summary_table.csv files found.")

        return {
            "paper_summary_all": pd.DataFrame(),
            "pivot": pd.DataFrame(),
            "figure": None,
            "run_log": run_log_df,
            "figures": {},
        }

    paper_summary_all = pd.concat(paper_summary_dfs, ignore_index=True)

    metric = "mean_features_for_80pct_late_layers"

    required_cols = [
        "model_label",
        "emotion",
        metric,
    ]

    missing_cols = [
        col for col in required_cols
        if col not in paper_summary_all.columns
    ]

    if len(missing_cols) > 0:
        raise ValueError(
            f"Missing required columns: {missing_cols}. "
            f"Available columns: {list(paper_summary_all.columns)}"
        )

    paper_summary_all[metric] = pd.to_numeric(
        paper_summary_all[metric],
        errors="coerce",
    )

    paper_summary_all = paper_summary_all.dropna(
        subset=["model_label", "emotion", metric]
    ).copy()

    pivot = paper_summary_all.pivot_table(
        index="model_label",
        columns="emotion",
        values=metric,
        aggfunc="mean",
    )

    pivot = pivot.reindex(
        columns=[emotion for emotion in emotions if emotion in pivot.columns]
    )

    if pivot.empty:
        print("Pivot table is empty. No figure generated.")

        return {
            "paper_summary_all": paper_summary_all,
            "pivot": pivot,
            "figure": None,
            "run_log": run_log_df,
            "figures": {},
        }

    # Clean up labels for display
    display_pivot = pivot.copy()
    display_pivot.index = [prettify_model_label(label) for label in display_pivot.index]
    display_pivot.columns = [str(x).capitalize() for x in display_pivot.columns]

    # --- MODEL REORDERING PROTOCOL ---
    sorted_index = sorted(display_pivot.index, key=make_sorting_key)
    display_pivot = display_pivot.reindex(sorted_index)

    # Set up paper typography (Clean sans-serif)
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    # Auto-adjust layout frames
    fig_height = max(4.0, 0.45 * len(display_pivot) + 1.0)
    fig_width = max(8.0, 0.95 * len(display_pivot.columns) + 2.5)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    # --- LOGARITHMIC SCALING CONTRAST ---
    # Since values live in a narrow band (2 to 7), a log norm stretches color variance
    raw_values = display_pivot.values
    valid_values = raw_values[~np.isnan(raw_values)]
    
    if len(valid_values) > 0:
        data_min = np.min(valid_values)
        data_max = np.max(valid_values)
        
        # Floor/ceil limits slightly to prevent clipping out pure whites/blacks
        # LogNorm requires values strictly greater than 0
        v_min_calibrated = max(0.1, data_min * 0.9)
        v_max_calibrated = data_max * 1.1
        
        log_norm = mpl.colors.LogNorm(vmin=v_min_calibrated, vmax=v_max_calibrated)
    else:
        log_norm = None

    # Render optimized seaborn layout using our custom Logarithmic norm
    sns.heatmap(
        display_pivot,
        ax=ax,
        cmap="Blues_r",  
        annot=True,
        fmt=".1f",
        linewidths=1.5,
        norm=log_norm,  # Injects the logarithmic color spacer directly
        cbar_kws={
            "fraction": 0.03, 
            "pad": 0.04
        },
        annot_kws={"fontsize": 9.5}
    )

    # Clean colorbar formatting using matplotlib's colorbar object hooked by seaborn
    cbar = ax.collections[0].colorbar
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(size=0)
    
    # Force clean scalar numbers on the color bar text instead of exponential scientific notations
    cbar.ax.yaxis.set_major_formatter(mpl.ticker.ScalarFormatter())
    cbar.set_label("# features for 80% effect", labelpad=12, fontsize=10, fontweight="medium")

    # Rotate column text slightly to match presentation standards
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    ax.set_yticklabels(ax.get_yticklabels(), fontweight="medium", rotation=0)

    # Strip messy axis tags
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(which="both", bottom=False, left=False)

    fig.tight_layout()

    figure_path = os.path.join(
        output_dir,
        "heatmap_late_layer_emotion_feature_concentration.png",
    )

    if save:
        fig.savefig(
            figure_path,
            dpi=dpi,
            bbox_inches="tight",
            facecolor="white",
        )
        print("Saved figure:", figure_path)

    if show:
        plt.show()
    else:
        plt.close(fig)

    figures = {
        "late_layer_emotion_feature_concentration": figure_path if save else None
    }

    return {
        "paper_summary_all": paper_summary_all,
        "pivot": pivot,
        "figure": figure_path if save else None,
        "run_log": run_log_df,
        "figures": figures,
    }

In [ ]:
MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

results = run_all_model_publishable_feature_aggregate_plot(
    models=MODELS,
    base_data_dir="data",
    output_dir="paper_visuals",
    skip_models={
        # Add skipped models here if needed.
    },
    emotions=DEFAULT_EMOTIONS,
    show=True,
    save=True,
    dpi=300,
)

paper_summary_all = results["paper_summary_all"]
concentration_pivot = results["pivot"]
figure_path = results["figure"]
run_log = results["run_log"]
figures = results["figures"]

print("Figure:", figure_path)
# display(concentration_pivot)

In [ ]:
# ============================================================
# Full replacement function (Original Logic + Clean Seaborn)
# ============================================================

def plot_all_model_swap_summary(
    models,
    base_data_dir="data",
    output_dir="paper_visuals",
    skip_models=None,
    emotions=DEFAULT_EMOTIONS,
    show=True,
    save=True,
    dpi=300,
    flip_target_col="flip_to_target_rate_clean_source_only",
):
    """
    Plot only one aggregate controllability figure:
        model x target emotion heatmap

    Metric:
        flip_to_target_rate_clean_source_only

    Meaning:
        For each source -> target pair, first select the best layer by flip-to-target rate.
        Then average over source emotions for each target emotion and model.

    Saves only:
        paper_visuals/heatmap_model_target_controllability.png
    """
    import os
    import re
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    import seaborn as sns

    if skip_models is None:
        skip_models = set()
    else:
        skip_models = set(skip_models)

    os.makedirs(output_dir, exist_ok=True)

    def get_sae_sizes_for_model(model_name):
        if model_name.startswith("google/gemma-2"):
            return [None]

        if model_name.startswith("meta-llama/"):
            return [None]

        if model_name.startswith("google/gemma-3"):
            return ["small", "big"]

        return [None]

    def get_model_slug(model_name):
        return model_name.replace("/", "-")

    def get_model_label(model_name, sae_size):
        slug = get_model_slug(model_name)

        if sae_size is None:
            return slug

        return f"{slug}-{sae_size}"

    def get_feature_dir(model_name, sae_size):
        slug = get_model_slug(model_name)

        if sae_size is None:
            return os.path.join(base_data_dir, f"{slug}-feature-act")

        return os.path.join(base_data_dir, f"{slug}-feature-act-{sae_size}")

    def prettify_model_label(label):
        label = label.replace("google-", "")
        label = label.replace("meta-llama-", "")
        label = label.replace("gemma-2-", "Gemma-2-")
        label = label.replace("gemma-3-", "Gemma-3-")
        label = label.replace("-small", " small")
        label = label.replace("-big", " big")
        return label

    # Custom size-sorting layout rule (Pushes Llama to the absolute base)
    def make_sorting_key(prettified_label):
        is_llama = 1 if "Llama" in prettified_label else 0
        match = re.search(r"(\d+(?:\.\d+)?)[Bb]", prettified_label)
        size = float(match.group(1)) if match else 0.0
        is_big_sae = 1 if "big" in prettified_label else 0
        return (is_llama, size, is_big_sae)

    all_dfs = []
    run_log = []

    for model_name in models:
        if model_name in skip_models:
            run_log.append(
                {
                    "model_name": model_name,
                    "sae_size": "none",
                    "model_label": get_model_label(model_name, None),
                    "status": "skipped",
                    "summary_path": "",
                }
            )
            continue

        for sae_size in get_sae_sizes_for_model(model_name):
            model_label = get_model_label(model_name, sae_size)
            feature_dir = get_feature_dir(model_name, sae_size)

            summary_path = os.path.join(
                feature_dir,
                "cross_emotion_layer_feature_swap",
                "cross_emotion_layer_feature_swap_summary.csv",
            )

            if not os.path.exists(summary_path):
                print("Missing:", summary_path)

                run_log.append(
                    {
                        "model_name": model_name,
                        "sae_size": sae_size if sae_size is not None else "none",
                        "model_label": model_label,
                        "status": "missing",
                        "summary_path": summary_path,
                    }
                )
                continue

            df_one = pd.read_csv(summary_path)

            df_one["model_name"] = model_name
            df_one["sae_size"] = sae_size if sae_size is not None else "none"
            df_one["model_label"] = model_label

            all_dfs.append(df_one)

            print("Loaded:", summary_path)

            run_log.append(
                {
                    "model_name": model_name,
                    "sae_size": sae_size if sae_size is not None else "none",
                    "model_label": model_label,
                    "status": "loaded",
                    "summary_path": summary_path,
                }
            )

    run_log_df = pd.DataFrame(run_log)

    if len(all_dfs) == 0:
        print("No swap summaries found.")

        return {
            "df": pd.DataFrame(),
            "best_pair_layer": pd.DataFrame(),
            "target_by_model": pd.DataFrame(),
            "pivot": pd.DataFrame(),
            "figure": None,
            "run_log": run_log_df,
            "figures": {},
        }

    df = pd.concat(all_dfs, ignore_index=True)

    required_cols = [
        "model_label",
        "source_emotion",
        "target_emotion",
        "layer_idx",
        flip_target_col,
    ]

    missing_cols = [
        col for col in required_cols
        if col not in df.columns
    ]

    if len(missing_cols) > 0:
        raise ValueError(
            f"Missing required columns: {missing_cols}. "
            f"Available columns: {list(df.columns)}"
        )

    df["layer_idx"] = pd.to_numeric(df["layer_idx"], errors="coerce")
    df[flip_target_col] = pd.to_numeric(df[flip_target_col], errors="coerce")

    df = df.dropna(
        subset=[
            "layer_idx",
            flip_target_col,
        ]
    ).copy()

    df["layer_idx"] = df["layer_idx"].astype(int)

    # Best layer for every model/source/target pair by flip-to-target.
    best_pair_layer = (
        df.sort_values(flip_target_col, ascending=False)
        .groupby(
            [
                "model_label",
                "source_emotion",
                "target_emotion",
            ],
            as_index=False,
        )
        .first()
    )

    # Aggregate controllability:
    # for each model and target emotion, average best-layer conversion over sources.
    target_by_model = (
        best_pair_layer
        .groupby(
            [
                "model_label",
                "target_emotion",
            ],
            as_index=False,
        )
        .agg(
            mean_best_flip_to_target=(flip_target_col, "mean"),
        )
    )

    pivot = best_pair_layer.pivot_table(
        index="model_label",
        columns="target_emotion",
        values=flip_target_col,
        aggfunc="mean",
    )

    pivot = pivot.reindex(
        columns=[
            emotion for emotion in emotions
            if emotion in pivot.columns
        ]
    )

    if pivot.empty:
        print("Pivot table is empty. No figure generated.")

        return {
            "df": df,
            "best_pair_layer": best_pair_layer,
            "target_by_model": target_by_model,
            "pivot": pivot,
            "figure": None,
            "run_log": run_log_df,
            "figures": {},
        }

    # Format matrix and sort logically by size
    display_pivot = pivot.copy()
    display_pivot.index = [prettify_model_label(label) for label in display_pivot.index]
    display_pivot.columns = [str(x).capitalize() for x in display_pivot.columns]

    sorted_index = sorted(display_pivot.index, key=make_sorting_key)
    display_pivot = display_pivot.reindex(sorted_index)

    # Clean sans-serif design properties for publication layouts
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    fig_height = max(4.0, 0.45 * len(display_pivot) + 1.0)
    fig_width = max(8.0, 0.95 * len(display_pivot.columns) + 2.5)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    # --- SPREAD CONTRAST BOOST ---
    raw_values = display_pivot.values
    valid_values = raw_values[~np.isnan(raw_values)]
    
    if len(valid_values) > 0:
        data_min = np.min(valid_values)
        data_max = np.max(valid_values)
        # Dynamic 5% range padding
        v_min_calibrated = max(0.0, data_min - 0.05)
        v_max_calibrated = min(1.0, data_max + 0.05)
    else:
        v_min_calibrated = 0.0
        v_max_calibrated = 1.0

    # Build clean Seaborn structure mapping lower values to dark blues
    sns.heatmap(
        display_pivot,
        ax=ax,
        cmap="Blues_r",  
        annot=True,
        fmt=".0%",
        linewidths=1.5,
        vmin=v_min_calibrated,
        vmax=v_max_calibrated,
        cbar_kws={
            "fraction": 0.03, 
            "pad": 0.04
        },
        annot_kws={"fontsize": 9.5}
    )

    cbar = ax.collections[0].colorbar
    cbar.ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f"{x:.0%}"))
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(size=0)
    cbar.set_label("Mean best-layer flip-to-target rate", labelpad=12, fontsize=10, fontweight="medium")

    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    ax.set_yticklabels(ax.get_yticklabels(), fontweight="medium", rotation=0)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(which="both", bottom=False, left=False)

    fig.tight_layout()

    figure_path = os.path.join(
        output_dir,
        "heatmap_model_target_controllability.png",
    )

    if save:
        fig.savefig(
            figure_path,
            dpi=dpi,
            bbox_inches="tight",
            facecolor="white",
        )
        print("Saved figure:", figure_path)

    if show:
        plt.show()
    else:
        plt.close(fig)

    figures = {
        "heatmap_model_target_controllability": figure_path if save else None
    }

    return {
        "df": df,
        "best_pair_layer": best_pair_layer,
        "target_by_model": target_by_model,
        "pivot": pivot,
        "figure": figure_path if save else None,
        "run_log": run_log_df,
        "figures": figures,
    }

In [ ]:
MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

results = plot_all_model_swap_summary(
    models=MODELS,
    base_data_dir="data",
    output_dir="paper_visuals",
    skip_models={
        # Add skipped models here if needed.
    },
    emotions=DEFAULT_EMOTIONS,
    show=True,
    save=True,
    dpi=300,
    flip_target_col="flip_to_target_rate_clean_source_only",
)

swap_all = results["df"]
best_pair_layer = results["best_pair_layer"]
target_by_model = results["target_by_model"]
controllability_pivot = results["pivot"]
figure_path = results["figure"]
run_log = results["run_log"]
figures = results["figures"]

print("Figure:", figure_path)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from sae_emotion.visual_utils import (
    run_emotion_overlap_matrix_from_affected,
    find_logit_cols,
)

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

MODELS = [
    {
        "name": "google-gemma-2-2b",
        "input_path": "data/google-gemma-2-2b-feature-act/auto_selected_unique_features_with_affected_emotions.csv",
        "output_dir": "paper_visuals/google-gemma-2-2b_overlap",
    },
    {
        "name": "google-gemma-3-12b-pt-big",
        "input_path": "data/google-gemma-3-12b-pt-feature-act-big/auto_selected_unique_features_with_affected_emotions.csv",
        "output_dir": "paper_visuals/google-gemma-3-12b-pt-big_overlap",
    },
]

for model_cfg in MODELS:
    print("=" * 80)
    print(model_cfg["name"])
    print("=" * 80)

    os.makedirs(model_cfg["output_dir"], exist_ok=True)

    df = pd.read_csv(model_cfg["input_path"])
    logit_cols = find_logit_cols(df, emotions=EMOTIONS)

    # Extract metrics from your utility engine
    overlap_df, jaccard_mat, overlap_coef_mat, count_mat = (
        run_emotion_overlap_matrix_from_affected(
            df,
            emotions=EMOTIONS,
            output_dir=model_cfg["output_dir"],
        )
    )

    # --- NEW HIGH-CONTRAST SANS-SERIF PRESENTATION ---
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    fig, ax = plt.subplots(figsize=(6.5, 5.5))

    # Clean up matrix axes string labels for presentation
    display_df = (
        jaccard_mat
        .reindex(index=EMOTIONS, columns=EMOTIONS)
        .copy()
    )
    display_df.index = [str(x).capitalize() for x in display_df.index]
    display_df.columns = [str(x).capitalize() for x in display_df.columns]

    # --- MASK DIAGONAL (REMOVE 1.00 CELLS) ---
    # This keeps the colorbar beautifully constrained to the 0-0.30 range
    mask = np.eye(len(display_df), dtype=bool)

    # --- CUSTOM SEGMENTED CONTRAST BOUNDARIES ---
    boundaries = [0.0, 0.02, 0.04, 0.06, 0.09, 0.12, 0.16, 0.20, 0.25, 0.30]
    
    base_cmap = plt.get_cmap("Blues")
    custom_colors = base_cmap(np.linspace(0.05, 0.95, len(boundaries) - 1))
    segmented_cmap = mpl.colors.ListedColormap(custom_colors)
    
    # Set the background color for the masked diagonal cells (soft neutral gray)
    segmented_cmap.set_bad(color="#f4f4f4")
    segmented_norm = mpl.colors.BoundaryNorm(boundaries, segmented_cmap.N)

    # Plot using our adjusted color spacing and diagonal mask
    sns.heatmap(
        display_df,
        ax=ax,
        mask=mask,
        cmap=segmented_cmap,
        norm=segmented_norm,
        annot=True,
        fmt=".2f",
        linewidths=1.5,
        cbar_kws={"fraction": 0.035, "pad": 0.04},
        annot_kws={"fontsize": 9.5}
    )

    # --- CLEAN COLORBAR TICK CONSTRAINTS ---
    cbar = ax.collections[0].colorbar
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(size=0)
    
    # Force the colorbar y-ticks to display only the absolute minimum and maximum values
    cbar.set_ticks([boundaries[0], boundaries[-1]])
    cbar.ax.set_yticklabels([f"{boundaries[0]:.2f}", f"{boundaries[-1]:.2f}"])
    
    cbar.set_label("Jaccard Overlap Index", labelpad=12, fontsize=10, fontweight="medium")

    # Rotate label tokens clean
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    ax.set_yticklabels(ax.get_yticklabels(), fontweight="medium", rotation=0)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(which="both", bottom=False, left=False)

    fig.tight_layout()

    # Save output to path
    figure_path = os.path.join(model_cfg["output_dir"], "jaccard_overlap_matrix.png")
    plt.savefig(figure_path, dpi=300, bbox_inches="tight", facecolor="white")
    print("Saved figure:", figure_path)
    
    plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# Config
# ============================================================

MODEL_RUNS = [
    ("google/gemma-2-2b", None),
    ("google/gemma-2-9b", None),

    ("google/gemma-3-4b-pt", "big"),
    ("google/gemma-3-4b-pt", "small"),
    ("google/gemma-3-4b-it", "big"),
    ("google/gemma-3-4b-it", "small"),

    ("meta-llama/Llama-3.1-8B", None),

    ("google/gemma-3-12b-pt", "small"),
    ("google/gemma-3-12b-it", "small"),
    ("google/gemma-3-12b-it", "big"),
    ("google/gemma-3-12b-pt", "big"),
]

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

OUT_DIR = "paper_visuals/emotion_ensemble_strength_corr_aggregate"
os.makedirs(OUT_DIR, exist_ok=True)

USE_CORRECT_ONLY = True
SHOW = True


# ============================================================
# Path helpers
# ============================================================

def model_slug(model_name):
    return model_name.replace("/", "-")


def run_name(model_name, sae_size):
    slug = model_slug(model_name)
    if sae_size is None:
        return slug
    return f"{slug}-{sae_size}"


def feature_dir_from_model(model_name, sae_size):
    slug = model_slug(model_name)

    if sae_size is None:
        return f"data/{slug}-feature-act"

    return f"data/{slug}-feature-act-{sae_size}"


def pred_path_from_model(model_name):
    slug = model_slug(model_name)
    return f"data/combined_zeroshot_preds_{slug}.csv"


def act_path_from_model(model_name, sae_size):
    slug = model_slug(model_name)

    if "gemma-3" in model_name and sae_size is not None:
        return f"data/sae_acts_combined_zeroshot_{slug}_{sae_size}.npy"

    return f"data/sae_acts_combined_zeroshot_{slug}.npy"


def feature_path_from_model(model_name, sae_size):
    feature_dir = feature_dir_from_model(model_name, sae_size)
    return os.path.join(
        feature_dir,
        "auto_selected_unique_features_with_affected_emotions.csv",
    )


# ============================================================
# Helpers
# ============================================================

def parse_emotions(x):
    if pd.isna(x):
        return set()

    return set(
        v.strip().lower()
        for v in str(x).split(",")
        if v.strip()
    )


def safe_corr(x, y):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) == 0:
        return np.nan

    if np.std(x) < 1e-8 or np.std(y) < 1e-8:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])


def safe_save_csv(df, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_csv(path, index=True)
    print("Saved:", path)


def savefig(path, show=True):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)

    if show:
        plt.show()
    else:
        plt.close()


# ============================================================
# Core calculation
# ============================================================

def build_emotion_ensemble_mean_activation(
    pred_path,
    act_path,
    feature_path,
    emotions=EMOTIONS,
    use_correct_only=True,
):
    pred_df = pd.read_csv(pred_path)
    acts = np.load(act_path, mmap_mode="r")
    feature_df = pd.read_csv(feature_path)

    required_pred_cols = ["input_text", "emotion", "pred_emotion"]
    missing_pred = [c for c in required_pred_cols if c not in pred_df.columns]
    if missing_pred:
        raise ValueError(f"Missing prediction columns in {pred_path}: {missing_pred}")

    required_feature_cols = ["layer_idx", "feature_idx", "affected_emotions"]
    missing_feature = [c for c in required_feature_cols if c not in feature_df.columns]
    if missing_feature:
        raise ValueError(f"Missing feature columns in {feature_path}: {missing_feature}")

    pred_df["emotion"] = pred_df["emotion"].astype(str).str.lower().str.strip()
    pred_df["pred_emotion"] = pred_df["pred_emotion"].astype(str).str.lower().str.strip()

    pred_df = pred_df[pred_df["emotion"].isin(emotions)].copy()
    pred_df = pred_df[pred_df["pred_emotion"].isin(emotions)].copy()

    if use_correct_only:
        pred_df = pred_df[pred_df["emotion"] == pred_df["pred_emotion"]].copy()

    pred_df = pred_df.reset_index(drop=False).rename(
        columns={"index": "original_instance_idx"}
    )

    instance_indices = pred_df["original_instance_idx"].astype(int).to_numpy()

    feature_df["layer_idx"] = feature_df["layer_idx"].astype(int)
    feature_df["feature_idx"] = feature_df["feature_idx"].astype(int)
    feature_df["affected_emotion_set"] = feature_df["affected_emotions"].apply(
        parse_emotions
    )

    feature_df = feature_df.drop_duplicates(
        subset=["layer_idx", "feature_idx"]
    ).copy()

    wide_mean = pd.DataFrame(index=instance_indices)

    feature_count_rows = []

    for emotion in emotions:
        sub = feature_df[
            feature_df["affected_emotion_set"].apply(lambda s: emotion in s)
        ].copy()

        feature_list = list(
            zip(
                sub["layer_idx"].astype(int).tolist(),
                sub["feature_idx"].astype(int).tolist(),
            )
        )

        feature_count_rows.append(
            {
                "emotion": emotion,
                "n_features": len(feature_list),
            }
        )

        if len(feature_list) == 0:
            wide_mean[emotion] = np.nan
            continue

        feature_acts = []

        for layer_idx, feature_idx in feature_list:
            vec = np.asarray(
                acts[int(layer_idx), instance_indices, int(feature_idx)],
                dtype=np.float32,
            )
            feature_acts.append(vec)

        feature_acts = np.stack(feature_acts, axis=0)
        wide_mean[emotion] = feature_acts.mean(axis=0)

    meta = pred_df.set_index("original_instance_idx")[
        ["emotion", "pred_emotion", "input_text"]
    ]

    wide_mean = wide_mean.join(meta)

    feature_count_df = pd.DataFrame(feature_count_rows)

    return wide_mean, feature_count_df


def compute_strength_corr_matrix(wide_mean, emotions=EMOTIONS):
    corr_mat = pd.DataFrame(
        index=emotions,
        columns=emotions,
        dtype=float,
    )

    for emotion_a in emotions:
        for emotion_b in emotions:
            a = wide_mean[emotion_a].astype(float).to_numpy()
            b = wide_mean[emotion_b].astype(float).to_numpy()

            corr_mat.loc[emotion_a, emotion_b] = safe_corr(a, b)

    return corr_mat


def matrix_to_long(corr_mat, model_name, sae_size, run):
    rows = []

    for emotion_a in corr_mat.index:
        for emotion_b in corr_mat.columns:
            rows.append(
                {
                    "model_name": model_name,
                    "sae_size": sae_size if sae_size is not None else "none",
                    "run": run,
                    "emotion_a": emotion_a,
                    "emotion_b": emotion_b,
                    "strength_corr": corr_mat.loc[emotion_a, emotion_b],
                }
            )

    return pd.DataFrame(rows)


# ============================================================
# Run all models
# ============================================================

all_corr_long = []
all_feature_counts = []
all_model_mats = {}

for model_name, sae_size in MODEL_RUNS:
    run = run_name(model_name, sae_size)

    pred_path = pred_path_from_model(model_name)
    act_path = act_path_from_model(model_name, sae_size)
    feature_path = feature_path_from_model(model_name, sae_size)

    model_out_dir = os.path.join(OUT_DIR, "per_model", run)
    os.makedirs(model_out_dir, exist_ok=True)

    print("\n" + "=" * 80)
    print("Run:", run)
    print("Prediction:", pred_path)
    print("Activation:", act_path)
    print("Feature:", feature_path)
    print("=" * 80)

    if not os.path.exists(pred_path):
        print("Skipped because prediction file does not exist:", pred_path)
        continue

    if not os.path.exists(act_path):
        print("Skipped because activation file does not exist:", act_path)
        continue

    if not os.path.exists(feature_path):
        print("Skipped because feature file does not exist:", feature_path)
        continue

    wide_mean, feature_count_df = build_emotion_ensemble_mean_activation(
        pred_path=pred_path,
        act_path=act_path,
        feature_path=feature_path,
        emotions=EMOTIONS,
        use_correct_only=USE_CORRECT_ONLY,
    )

    wide_mean_path = os.path.join(
        model_out_dir,
        "wide_emotion_ensemble_mean_activation.csv",
    )
    safe_save_csv(wide_mean, wide_mean_path)

    feature_count_df["model_name"] = model_name
    feature_count_df["sae_size"] = sae_size if sae_size is not None else "none"
    feature_count_df["run"] = run
    feature_count_path = os.path.join(
        model_out_dir,
        "emotion_feature_counts.csv",
    )
    safe_save_csv(feature_count_df, feature_count_path)

    corr_mat = compute_strength_corr_matrix(
        wide_mean,
        emotions=EMOTIONS,
    )

    corr_mat_path = os.path.join(
        model_out_dir,
        "emotion_ensemble_strength_correlation_matrix.csv",
    )
    safe_save_csv(corr_mat, corr_mat_path)

    corr_long = matrix_to_long(
        corr_mat=corr_mat,
        model_name=model_name,
        sae_size=sae_size,
        run=run,
    )

    corr_long_path = os.path.join(
        model_out_dir,
        "emotion_ensemble_strength_correlation_long.csv",
    )
    corr_long.to_csv(corr_long_path, index=False)
    print("Saved:", corr_long_path)

    plot_corr_heatmap(
        corr_mat=corr_mat,
        title=f"Correlation Between Emotion Ensemble Activation Strengths\n{run}",
        output_path=os.path.join(
            model_out_dir,
            "heatmap_activation_strength_correlation.png",
        ),
        emotions=EMOTIONS,
        show=False,
    )

    all_corr_long.append(corr_long)
    all_feature_counts.append(feature_count_df)
    all_model_mats[run] = corr_mat


# ============================================================
# Save all per-model values
# ============================================================

if len(all_corr_long) == 0:
    raise RuntimeError("No valid model results were computed. Check file paths.")

all_corr_df = pd.concat(all_corr_long, ignore_index=True)

all_corr_path = os.path.join(
    OUT_DIR,
    "all_models_emotion_ensemble_strength_correlation_long.csv",
)
all_corr_df.to_csv(all_corr_path, index=False)
print("Saved:", all_corr_path)

if len(all_feature_counts) > 0:
    all_feature_counts_df = pd.concat(all_feature_counts, ignore_index=True)
    all_feature_counts_path = os.path.join(
        OUT_DIR,
        "all_models_emotion_feature_counts.csv",
    )
    all_feature_counts_df.to_csv(all_feature_counts_path, index=False)
    print("Saved:", all_feature_counts_path)


# ============================================================
# Aggregate across models
# ============================================================

agg_df = (
    all_corr_df
    .groupby(["emotion_a", "emotion_b"], dropna=False)
    .agg(
        mean_strength_corr=("strength_corr", "mean"),
        std_strength_corr=("strength_corr", "std"),
        n_models=("strength_corr", "count"),
    )
    .reset_index()
)

agg_long_path = os.path.join(
    OUT_DIR,
    "aggregate_emotion_ensemble_strength_correlation_long.csv",
)
agg_df.to_csv(agg_long_path, index=False)
print("Saved:", agg_long_path)

agg_corr_mat = (
    agg_df
    .pivot(
        index="emotion_a",
        columns="emotion_b",
        values="mean_strength_corr",
    )
    .reindex(index=EMOTIONS, columns=EMOTIONS)
)

agg_std_mat = (
    agg_df
    .pivot(
        index="emotion_a",
        columns="emotion_b",
        values="std_strength_corr",
    )
    .reindex(index=EMOTIONS, columns=EMOTIONS)
)

agg_n_mat = (
    agg_df
    .pivot(
        index="emotion_a",
        columns="emotion_b",
        values="n_models",
    )
    .reindex(index=EMOTIONS, columns=EMOTIONS)
)

agg_corr_path = os.path.join(
    OUT_DIR,
    "aggregate_emotion_ensemble_strength_correlation_matrix.csv",
)
agg_std_path = os.path.join(
    OUT_DIR,
    "aggregate_emotion_ensemble_strength_correlation_std_matrix.csv",
)
agg_n_path = os.path.join(
    OUT_DIR,
    "aggregate_emotion_ensemble_strength_correlation_n_models_matrix.csv",
)

safe_save_csv(agg_corr_mat, agg_corr_path)
safe_save_csv(agg_std_mat, agg_std_path)
safe_save_csv(agg_n_mat, agg_n_path)

In [ ]:
def plot_corr_heatmap(
    corr_mat,
    title,
    output_path,
    emotions=EMOTIONS,
    show=True,
):
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    import seaborn as sns

    plot_mat = corr_mat.reindex(index=emotions, columns=emotions)

    # Format the row/column labels elegantly
    display_df = plot_mat.copy()
    display_df.index = [str(x).capitalize() for x in display_df.index]
    display_df.columns = [str(x).capitalize() for x in display_df.columns]

    # Clean sans-serif typography specifications for publications
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    # Auto-adjust frame constraints matching our size standards
    fig_height = max(4.5, 0.5 * len(display_df) + 1.0)
    fig_width = max(6.5, 0.95 * len(display_df.columns) + 1.5)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    # Mask out the diagonal identity loop (removes the constant 1.00 values)
    mask = np.eye(len(display_df), dtype=bool)

    # Diverging colormap choice: 'RdBu_r' (Red=Positive, Blue=Negative, Center=Neutral)
    cmap = plt.get_cmap("RdBu_r").copy()
    cmap.set_bad(color="#f4f4f4")  # Soft background tint for the identity diagonal mask

    # Render layout framework
    sns.heatmap(
        display_df,
        ax=ax,
        mask=mask,
        cmap=cmap,
        annot=True,
        fmt=".2f",
        linewidths=1.5,
        vmin=-1.0,  # Standardized absolute anchor bounds for Pearson matrices
        vmax=1.0,
        cbar_kws={"fraction": 0.035, "pad": 0.04},
        annot_kws={"fontsize": 9.5}
    )

    # --- CLEAN COLORBAR CONTRAST CONSTRAINTS ---
    cbar = ax.collections[0].colorbar
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(size=0)
    
    # Strip unnecessary midway intervals—only capture absolute bounds + center split
    cbar.set_ticks([-1.0, 0.0, 1.0])
    cbar.ax.set_yticklabels(["-1.00", "0.00", "1.00"])
    cbar.set_label("Pearson correlation", labelpad=12, fontsize=10, fontweight="medium")

    # Layout axis tuning adjustments
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    ax.set_yticklabels(ax.get_yticklabels(), fontweight="medium", rotation=0)

    # Remove dirty layout captions
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(which="both", bottom=False, left=False)

    if title:
        ax.set_title(title, pad=18, fontsize=11, fontweight="semibold", wrap=True)

    fig.tight_layout()

    # Pass logic execution cleanly back down to your downstream custom savefig code engine
    savefig(output_path, show=show)


# ============================================================
# Plot aggregate heatmap
# ============================================================

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]
OUT_DIR = "paper_visuals/emotion_ensemble_strength_corr_aggregate"

agg_corr_mat = pd.read_csv(
    os.path.join(
        OUT_DIR,
        "aggregate_emotion_ensemble_strength_correlation_matrix.csv",
    ),
    index_col=0,
)

plot_corr_heatmap(
    corr_mat=agg_corr_mat,
    title="",
    output_path=os.path.join(
        OUT_DIR,
        "aggregate_heatmap_activation_strength_correlation.png",
    ),
    emotions=EMOTIONS,
    show=True,
)

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from sae_emotion.visual_utils import (
    run_emotion_overlap_matrix_from_affected,
    find_logit_cols,
)

# ============================================================
# 1. SETUP & PARAMETERS
# ============================================================
EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]
OUT_DIR_CORR = "paper_visuals/emotion_ensemble_strength_corr_aggregate"
FINAL_VISUALS_DIR = "paper_visuals"
os.makedirs(FINAL_VISUALS_DIR, exist_ok=True)

MODELS_ABR = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

# Replaced gemma-3-12b with gemma-2-2b configuration parameters
MODELS_OVERLAP_CFG = [
    {
        "name": "google-gemma-2-2b",
        "input_path": "data/google-gemma-2-2b-feature-act/auto_selected_unique_features_with_affected_emotions.csv",
        "output_dir": "paper_visuals/google-gemma-2-2b_overlap",
    },
]

# ============================================================
# 2. RUN UPSTREAM METRIC CALCULATIONS
# ============================================================
results = run_all_model_publishable_feature_aggregate_plot(
    models=MODELS_ABR,
    base_data_dir="data",
    output_dir=FINAL_VISUALS_DIR,
    skip_models={},
    emotions=EMOTIONS,
    show=False, 
    save=False
)
concentration_pivot = results["pivot"]

overlap_data = []
for model_cfg in MODELS_OVERLAP_CFG:
    os.makedirs(model_cfg["output_dir"], exist_ok=True)
    df_overlap = pd.read_csv(model_cfg["input_path"])
    _ = find_logit_cols(df_overlap, emotions=EMOTIONS)
    
    _, jaccard_mat, _, _ = run_emotion_overlap_matrix_from_affected(
        df_overlap, emotions=EMOTIONS, output_dir=model_cfg["output_dir"]
    )
    overlap_data.append((model_cfg["name"], jaccard_mat))

agg_corr_mat = pd.read_csv(
    os.path.join(OUT_DIR_CORR, "aggregate_emotion_ensemble_strength_correlation_matrix.csv"),
    index_col=0,
)
corr_plot_mat = agg_corr_mat.reindex(index=EMOTIONS, columns=EMOTIONS)


# ============================================================
# 3. COMPOSITE SUBPLOT CANVAS RENDERING (DUAL INDEPENDENT GRIDS)
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig = plt.figure(figsize=(14, 5))

# Left Grid (Panel 1): Stays anchored from 0.05 to 0.41 (40% width target)
gs_left = mpl.gridspec.GridSpec(1, 1, left=0.05, right=0.41)
ax1 = fig.add_subplot(gs_left[0])

# Right Grid (Panels 2 & 3): Shifted left boundary from 0.48 to 0.50 to add a 4px tracking gap
# The trailing margin is pushed to 0.99 to keep the remaining cells square and clean
gs_right = mpl.gridspec.GridSpec(1, 2, left=0.50, right=0.99, wspace=0.18)
ax2 = fig.add_subplot(gs_right[0])
ax3 = fig.add_subplot(gs_right[1], sharey=ax2)

axes = [ax1, ax2, ax3]

# ------------------------------------------------------------
# PANEL 1: Late-Layer Feature Concentration Heatmap (40% Area)
# ------------------------------------------------------------
p1_pivot = concentration_pivot.copy()
p1_pivot.index = [prettify_model_label(x) for x in p1_pivot.index]
p1_pivot.columns = [str(x).capitalize() for x in p1_pivot.columns]

sorted_index = sorted(p1_pivot.index, key=make_sorting_key)
p1_pivot = p1_pivot.reindex(sorted_index)

p1_vals = p1_pivot.values
p1_valid = p1_vals[~np.isnan(p1_vals)]
p1_vmin = max(0.0, np.min(p1_valid) * 0.9) if len(p1_valid) > 0 else 1.0
p1_vmax = np.max(p1_valid) * 1.1 if len(p1_valid) > 0 else 10.0
p1_log_norm = mpl.colors.LogNorm(vmin=p1_vmin, vmax=p1_vmax)

sns.heatmap(
    p1_pivot, ax=ax1, cmap="Blues_r", annot=True, fmt=".1f", linewidths=1.2,
    norm=p1_log_norm, cbar_kws={"fraction": 0.04, "pad": 0.04}
)
cbar1 = ax1.collections[0].colorbar
cbar1.outline.set_visible(False)
cbar1.ax.tick_params(size=0)
cbar1.ax.yaxis.set_major_formatter(mpl.ticker.ScalarFormatter())
cbar1.set_label("# features", labelpad=-4, fontsize=11, fontweight="medium")
ax1.set_title("(A) # of Features", pad=12, fontsize=12, fontweight="normal")


# ------------------------------------------------------------
# PANEL 2: Jaccard Overlap Matrix (30% Area)
# ------------------------------------------------------------
boundaries = [0.0, 0.02, 0.04, 0.06, 0.09, 0.12, 0.16, 0.20, 0.25, 0.30]
base_cmap = plt.get_cmap("Blues")
custom_colors = base_cmap(np.linspace(0.05, 0.95, len(boundaries) - 1))
segmented_cmap = mpl.colors.ListedColormap(custom_colors)
segmented_cmap.set_bad(color="#f4f4f4")
segmented_norm = mpl.colors.BoundaryNorm(boundaries, segmented_cmap.N)

model_name, jaccard_mat = overlap_data[0]
p_df = (
    jaccard_mat
    .reindex(index=EMOTIONS, columns=EMOTIONS)
    .copy()
)

p_df.index = [str(x).capitalize() for x in p_df.index]
p_df.columns = [str(x).capitalize() for x in p_df.columns]

mask_diag = np.eye(len(p_df), dtype=bool)

sns.heatmap(
    p_df, ax=ax2, mask=mask_diag, cmap=segmented_cmap, norm=segmented_norm,
    annot=True, fmt=".2f", linewidths=1.2, cbar_kws={"fraction": 0.04, "pad": 0.04}
)
cbar_ov = ax2.collections[0].colorbar
cbar_ov.outline.set_visible(False)
cbar_ov.ax.tick_params(size=0)
cbar_ov.set_ticks([boundaries[0], boundaries[-1]])
cbar_ov.ax.set_yticklabels([f"{boundaries[0]:.2f}", f"{boundaries[-1]:.2f}"])
cbar_ov.set_label("Jaccard Overlap Index", labelpad=-4, fontsize=11, fontweight="medium")

# Title adjusted to plain-weight Gemma-2-2B mapping constraints
ax2.set_title("(B) Gemma-2-2B Feature Overlap", pad=12, fontsize=12, fontweight="normal")


# ------------------------------------------------------------
# PANEL 3: Mean Activation Strength Pearson Correlation (30% Area)
# ------------------------------------------------------------
p4_df = corr_plot_mat.copy()
p4_df.index = [str(x).capitalize() for x in p4_df.index]
p4_df.columns = [str(x).capitalize() for x in p4_df.columns]

mask_corr = np.eye(len(p4_df), dtype=bool)
cmap_corr = plt.get_cmap("RdBu_r").copy()
cmap_corr.set_bad(color="#f4f4f4")

sns.heatmap(
    p4_df, ax=ax3, mask=mask_corr, cmap=cmap_corr, annot=True, fmt=".2f", linewidths=1.2,
    vmin=-1.0, vmax=1.0, cbar_kws={"fraction": 0.04, "pad": 0.04}
)
cbar4 = ax3.collections[0].colorbar
cbar4.outline.set_visible(False)
cbar4.ax.tick_params(size=0)
cbar4.set_ticks([-1.0, 0.0, 1.0])
cbar4.ax.set_yticklabels(["-1.00", "0.00", "1.00"])
cbar4.set_label("Pearson correlation", labelpad=-4, fontsize=11, fontweight="medium")
ax3.set_title("(C) Activation Correlation", pad=12, fontsize=12, fontweight="normal")


# ============================================================
# 4. SHARED AXIS CLEANUP & EXPORT
# ============================================================
for idx, ax in enumerate(axes):
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(which="both", bottom=False, left=False)
    
    if idx > 1:
        plt.setp(ax.get_yticklabels(), visible=False)
        ax.yaxis.set_ticks_position('none')
    else:
        ax.set_yticklabels(ax.get_yticklabels(), fontweight="medium", rotation=0)

composite_output_path = os.path.join(FINAL_VISUALS_DIR, "composite_paper_isolated_narrow_report.png")
plt.savefig(composite_output_path, dpi=300, bbox_inches="tight", facecolor="white")
print("\n" + "="*50)
print("SUCCESS: Subplot widths precisely normalized to 40% / 30% / 30% configuration limits.")
print("Saved to:", composite_output_path)
print("="*50)

plt.show()

In [ ]:
import os
import re
import ast
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from sae_emotion.visual_utils import (
    run_all_model_publishable_feature_aggregate_plot,
    find_logit_cols,
)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ============================================================
# 1. SETUP & PARAMETERS
# ============================================================
EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]
OUT_DIR_CORR = "paper_visuals/emotion_ensemble_strength_corr_aggregate"
FINAL_VISUALS_DIR = "paper_visuals"
os.makedirs(FINAL_VISUALS_DIR, exist_ok=True)

MODELS_ABR = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]


def get_sae_sizes_for_model(model_name):
    if model_name.startswith("google/gemma-2"):
        return [None]
    if model_name.startswith("meta-llama/"):
        return [None]
    if model_name.startswith("google/gemma-3"):
        return ["small", "big"]
    return [None]


def get_model_slug(model_name):
    return model_name.replace("/", "-")


def get_model_label(model_name, sae_size):
    slug = get_model_slug(model_name)
    if sae_size is None:
        return slug
    return f"{slug}-{sae_size}"


def get_feature_dir(model_name, sae_size, base_data_dir="data"):
    slug = get_model_slug(model_name)
    if sae_size is None:
        return f"{base_data_dir}/{slug}-feature-act"
    return f"{base_data_dir}/{slug}-feature-act-{sae_size}"


def parse_affected_emotions_local(x):
    if pd.isna(x):
        return []
    x = str(x).strip()
    x = x.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    if x == "":
        return []
    parsed = [e.strip() for e in x.split(",")]
    return [e for e in EMOTIONS if e in parsed]


def compute_avg_logit_effect_per_feature(models, base_data_dir="data"):
    rows = []

    for model_name in models:
        for sae_size in get_sae_sizes_for_model(model_name):
            model_label = get_model_label(model_name, sae_size)
            feature_dir = get_feature_dir(model_name, sae_size, base_data_dir)
            feature_path = os.path.join(
                feature_dir,
                "auto_selected_unique_features_with_affected_emotions.csv",
            )

            if not os.path.exists(feature_path):
                print("Missing feature file:", feature_path)
                continue

            df = pd.read_csv(feature_path)
            logit_cols = find_logit_cols(df, emotions=EMOTIONS)

            df["affected_emotion_list"] = df["affected_emotions"].apply(
                parse_affected_emotions_local
            )

            for emotion in EMOTIONS:
                if emotion not in logit_cols:
                    rows.append({
                        "model_label": model_label,
                        "emotion": emotion,
                        "avg_abs_logit_effect_per_feature": np.nan,
                    })
                    continue

                col = logit_cols[emotion]
                mask = df["affected_emotion_list"].apply(lambda xs: emotion in xs)
                vals = pd.to_numeric(df.loc[mask, col], errors="coerce").abs()

                rows.append({
                    "model_label": model_label,
                    "emotion": emotion,
                    "avg_abs_logit_effect_per_feature": vals.mean(),
                })

    out = pd.DataFrame(rows)

    return out.pivot_table(
        index="model_label",
        columns="emotion",
        values="avg_abs_logit_effect_per_feature",
        aggfunc="mean",
    )


# ============================================================
# 2. RUN UPSTREAM METRIC CALCULATIONS
# ============================================================
results = run_all_model_publishable_feature_aggregate_plot(
    models=MODELS_ABR,
    base_data_dir="data",
    output_dir=FINAL_VISUALS_DIR,
    skip_models={},
    emotions=EMOTIONS,
    show=False,
)

paper_summary_all = results["paper_summary_all"]
overlap_all = results["overlap_all"]

concentration_pivot = paper_summary_all.pivot_table(
    index="model_label",
    columns="emotion",
    values="mean_features_for_80pct_late_layers",
    aggfunc="mean",
)
concentration_pivot = concentration_pivot.reindex(columns=EMOTIONS)

original_model_order = list(
    results["paper_summary_all"]["model_label"].drop_duplicates()
)

# move llama models to end while preserving all other order
llama_models = [
    x for x in original_model_order
    if "llama" in x.lower()
]

non_llama_models = [
    x for x in original_model_order
    if "llama" not in x.lower()
]

final_model_order = non_llama_models + llama_models

concentration_pivot = concentration_pivot.reindex(final_model_order)

avg_logit_pivot = compute_avg_logit_effect_per_feature(
    models=MODELS_ABR,
    base_data_dir="data",
)
avg_logit_pivot = avg_logit_pivot.reindex(
    index=concentration_pivot.index,
    columns=concentration_pivot.columns,
)

avg_overlap = (
    overlap_all
    .groupby(["emotion_1", "emotion_2"], as_index=False)["jaccard"]
    .mean()
)

jaccard_mat = avg_overlap.pivot(
    index="emotion_1",
    columns="emotion_2",
    values="jaccard",
)
jaccard_mat = jaccard_mat.reindex(index=EMOTIONS, columns=EMOTIONS)

agg_corr_mat = pd.read_csv(
    os.path.join(OUT_DIR_CORR, "aggregate_emotion_ensemble_strength_correlation_matrix.csv"),
    index_col=0,
)
corr_plot_mat = agg_corr_mat.reindex(index=EMOTIONS, columns=EMOTIONS)


# ============================================================
# 3. COMPOSITE SUBPLOT CANVAS RENDERING
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig = plt.figure(figsize=(14, 5))

# Squeeze main plot A width by 20%
# Old width: 0.41 - 0.05 = 0.36
# New width: 0.36 * 0.8 = 0.288, so right = 0.05 + 0.288 = 0.338
gs_left = mpl.gridspec.GridSpec(1, 1, left=0.05, right=0.338)
ax1 = fig.add_subplot(gs_left[0])

gs_right = mpl.gridspec.GridSpec(1, 2, left=0.462, right=0.99, wspace=0.18)
ax2 = fig.add_subplot(gs_right[0])
ax3 = fig.add_subplot(gs_right[1], sharey=ax2)

axes = [ax1, ax2, ax3]

# ============================================================
# PANEL 1: Refined Bubble Plot (Color=Logit, Size=Count)
# ============================================================

p1_pivot = concentration_pivot.copy()
p1_effect = avg_logit_pivot.copy()

p1_pivot.columns = [str(x).capitalize() for x in p1_pivot.columns]
p1_effect.columns = [str(x).capitalize() for x in p1_effect.columns]

y_coords, x_coords = np.indices(p1_pivot.shape)
x_flat = x_coords.flatten() + 0.5
y_flat = y_coords.flatten() + 0.5

size_vals = p1_pivot.values.flatten()
color_vals = p1_effect.values.flatten()

min_s, max_s = np.nanmin(size_vals), np.nanmax(size_vals)
point_sizes = ((size_vals - min_s) / (max_s - min_s) * 500) + 50

ax1.set_facecolor("white")
ax1.set_xlim(0, p1_pivot.shape[1])
ax1.set_ylim(p1_pivot.shape[0], 0)

for spine in ax1.spines.values():
    spine.set_visible(False)

ax1.set_xticks(np.arange(p1_pivot.shape[1]) + 0.5)
ax1.set_yticks(np.arange(p1_pivot.shape[0]) + 0.5)
ax1.set_xticklabels(p1_pivot.columns, rotation=30, ha="right")
ax1.set_yticklabels(p1_pivot.index)
ax1.grid(True, linestyle=":", alpha=0.4, zorder=1)

# Make values around 0.6–2 already appear dark
color_norm = mpl.colors.Normalize(vmin=0.0, vmax=1.5)

scatter = ax1.scatter(
    x_flat,
    y_flat,
    s=point_sizes,
    c=color_vals,
    cmap="Blues",
    norm=color_norm,
    edgecolors="none",
    linewidths=0,
    zorder=3,
)

ax_cbar = inset_axes(
    ax1,
    width="5%",
    height="40%",
    loc="upper left",
    bbox_to_anchor=(1.05, 0.55, 1, 0.4),
    bbox_transform=ax1.transAxes,
    borderpad=0,
)

cbar = fig.colorbar(scatter, cax=ax_cbar, orientation="vertical")
cbar.outline.set_visible(False)
cbar.ax.tick_params(size=0)
cbar.set_label("Avg. Logit Effect", fontsize=10)

handles, labels = [], []
size_steps = [min_s, (min_s + max_s) / 2, max_s]

for s_val in size_steps:
    s_mapped = ((s_val - min_s) / (max_s - min_s) * 500) + 50
    handles.append(
        plt.scatter(
            [],
            [],
            s=s_mapped,
            color="#555555",
            alpha=0.6,
            edgecolors="none",
        )
    )
    labels.append(f"{int(s_val)}")

leg = ax1.legend(
    handles,
    labels,
    title="# Features",
    loc="lower left",
    bbox_to_anchor=(1.05, 0.05),
    fontsize=9,
    frameon=False,
    labelspacing=1.2,
    handletextpad=1.0,
)
leg._legend_box.align = "left"

ax1.set_title("(A) Feature Concentration & Magnitude", pad=20, fontsize=12)


# ------------------------------------------------------------
# PANEL 2: Average Jaccard Overlap Matrix Across Models
# ------------------------------------------------------------
boundaries = [0.0, 0.02, 0.04, 0.06, 0.09, 0.12, 0.16, 0.20, 0.25, 0.30]
base_cmap = plt.get_cmap("Blues")
custom_colors = base_cmap(np.linspace(0.05, 0.95, len(boundaries) - 1))
segmented_cmap = mpl.colors.ListedColormap(custom_colors)
segmented_cmap.set_bad(color="#f4f4f4")
segmented_norm = mpl.colors.BoundaryNorm(boundaries, segmented_cmap.N)

p_df = jaccard_mat.copy()
p_df.index = [str(x).capitalize() for x in p_df.index]
p_df.columns = [str(x).capitalize() for x in p_df.columns]

mask_diag = np.eye(len(p_df), dtype=bool)

sns.heatmap(
    p_df,
    ax=ax2,
    mask=mask_diag,
    cmap=segmented_cmap,
    norm=segmented_norm,
    annot=True,
    fmt=".2f",
    linewidths=1.2,
    cbar_kws={"fraction": 0.04, "pad": 0.04},
)

cbar_ov = ax2.collections[0].colorbar
cbar_ov.outline.set_visible(False)
cbar_ov.ax.tick_params(size=0)
cbar_ov.set_ticks([boundaries[0], boundaries[-1]])
cbar_ov.ax.set_yticklabels([f"{boundaries[0]:.2f}", f"{boundaries[-1]:.2f}"])
cbar_ov.set_label("Jaccard Overlap Index", labelpad=-10, fontsize=11, fontweight="medium")
ax2.set_title("(B) Mean Feature Overlap", pad=12, fontsize=12, fontweight="normal")


# ------------------------------------------------------------
# PANEL 3: Mean Activation Strength Pearson Correlation
# ------------------------------------------------------------
p4_df = corr_plot_mat.copy()
p4_df.index = [str(x).capitalize() for x in p4_df.index]
p4_df.columns = [str(x).capitalize() for x in p4_df.columns]

mask_corr = np.eye(len(p4_df), dtype=bool)
cmap_corr = plt.get_cmap("RdBu_r").copy()
cmap_corr.set_bad(color="#f4f4f4")

sns.heatmap(
    p4_df,
    ax=ax3,
    mask=mask_corr,
    cmap=cmap_corr,
    annot=True,
    fmt=".2f",
    linewidths=1.2,
    vmin=-1.0,
    vmax=1.0,
    cbar_kws={"fraction": 0.04, "pad": 0.04},
)

cbar4 = ax3.collections[0].colorbar
cbar4.outline.set_visible(False)
cbar4.ax.tick_params(size=0)

# Removed the middle 0.00 tick label
cbar4.set_ticks([-1.0, 1.0])
cbar4.ax.set_yticklabels(["-1.00", "1.00"])

cbar4.set_label("Pearson correlation", labelpad=-10, fontsize=11, fontweight="medium")
ax3.set_title("(C) Activation Correlation", pad=12, fontsize=12, fontweight="normal")


# ============================================================
# 4. SHARED AXIS CLEANUP & EXPORT
# ============================================================
for idx, ax in enumerate(axes):
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(which="both", bottom=False, left=False)

    if idx > 1:
        plt.setp(ax.get_yticklabels(), visible=False)
        ax.yaxis.set_ticks_position("none")
    else:
        ax.set_yticklabels(ax.get_yticklabels(), fontweight="medium", rotation=0)

composite_output_path = os.path.join(
    FINAL_VISUALS_DIR,
    "composite_paper_isolated_narrow_report.png",
)

plt.savefig(composite_output_path, dpi=300, bbox_inches="tight", facecolor="white")

print("\n" + "=" * 50)
print("SUCCESS: Saved updated composite figure.")
print("Saved to:", composite_output_path)
print("=" * 50)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ============================================================
# Load
# ============================================================

CSV_PATH = (
    "data/"
    "google-gemma-2-9b-feature-act/"
    "per_layer_per_emotion_feature_ablation/"
    "per_layer_per_emotion_ablation_summary.csv"
)

OUT_DIR = "paper_visuals"
os.makedirs(OUT_DIR, exist_ok=True)

EMOTIONS = [
    "joy",
    "surprise",
    "disgust",
    "anger",
    "sadness",
    "fear",
]

df = pd.read_csv(CSV_PATH)

# ============================================================
# Keep only needed columns
# ============================================================

plot_df = df[
    [
        "target_emotion",
        "layer_idx",
        "prediction_flip_rate",
    ]
].copy()

plot_df["target_emotion"] = (
    plot_df["target_emotion"]
    .astype(str)
    .str.lower()
    .str.strip()
)

plot_df = plot_df[
    plot_df["target_emotion"].isin(EMOTIONS)
].copy()

# ============================================================
# Plot
# ============================================================

plt.figure(figsize=(8, 5))

sns.lineplot(
    data=plot_df,
    x="layer_idx",
    y="prediction_flip_rate",
    hue="target_emotion",
    hue_order=EMOTIONS,
    marker="o",
    linewidth=2,
)

plt.xlabel("Layer")
plt.ylabel("Prediction flip rate")
plt.title("Layer-wise Prediction Flip Rate")
plt.legend(title="Emotion")

plt.tight_layout()

out_path = os.path.join(
    OUT_DIR,
    "gemma2_9b_prediction_flip_rate_per_emotion.png",
)

plt.savefig(
    out_path,
    dpi=300,
    bbox_inches="tight",
)

print("Saved:", out_path)

plt.show()

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# ============================================================
# 1. SETUP & DATA LOADING
# ============================================================
EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]
FINAL_VISUALS_DIR = "paper_visuals"
os.makedirs(FINAL_VISUALS_DIR, exist_ok=True)

MODELS = [
    "google/gemma-2-2b", "google/gemma-2-9b", "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt", "google/gemma-3-4b-it", "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it", "google/gemma-3-12b-pt", "google/gemma-3-12b-it",
]

# --- Load Panel A Data (Line Plot) ---
CSV_PATH_A = "data/google-gemma-2-9b-feature-act/per_layer_per_emotion_feature_ablation/per_layer_per_emotion_ablation_summary.csv"
df_a = pd.read_csv(CSV_PATH_A)
plot_df_a = df_a[["target_emotion", "layer_idx", "prediction_flip_rate"]].copy()
plot_df_a["target_emotion"] = plot_df_a["target_emotion"].astype(str).str.lower().str.strip()
plot_df_a = plot_df_a[plot_df_a["target_emotion"].isin(EMOTIONS)].copy()

# --- Load Heatmap Data ---
res_b = plot_all_model_ablation_summary(models=MODELS, base_data_dir="data", output_dir=FINAL_VISUALS_DIR, show=False, save=False)
pivot_b = res_b["pivot"]

res_c = plot_all_model_swap_summary(models=MODELS, base_data_dir="data", output_dir=FINAL_VISUALS_DIR, show=False, save=False)
pivot_c = res_c["pivot"]

# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================
def prettify_model_label(label):
    label = label.replace("google-", "").replace("meta-llama-", "")
    label = label.replace("gemma-2-", "Gemma-2-").replace("gemma-3-", "Gemma-3-")
    label = label.replace("-feature-act", "").replace("-small", " (small)").replace("-big", " (big)")
    return label

def make_sorting_key(prettified_label):
    is_llama = 1 if "Llama" in prettified_label else 0
    match = re.search(r"(\d+(?:\.\d+)?)[Bb]", prettified_label)
    size = float(match.group(1)) if match else 0.0
    is_big_sae = 1 if "big" in prettified_label else 0
    return (is_llama, size, is_big_sae)

# ============================================================
# 3. COMPOSITE RENDERING
# ============================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig = plt.figure(figsize=(14, 5))

# --- Width Partitioning ---
# Grid 1: Panel A (Left)
gs_left = mpl.gridspec.GridSpec(1, 1, left=0.05, right=0.28)
ax1 = fig.add_subplot(gs_left[0])

# Grid 2: Panels B & C (Right) 
# Reduced left from 0.52 to 0.48 to decrease distance by ~10 pixels
gs_right = mpl.gridspec.GridSpec(1, 2, left=0.42, right=0.98, wspace=0.15)
ax2 = fig.add_subplot(gs_right[0])
ax3 = fig.add_subplot(gs_right[1], sharey=ax2)

axes = [ax1, ax2, ax3]

# --- PANEL A: Desaturated Categorical Hues ---
# Using 'Set2' for a professional, desaturated categorical palette
desaturated_palette = sns.color_palette("Set2", n_colors=len(EMOTIONS))

sns.lineplot(
    data=plot_df_a, x="layer_idx", y="prediction_flip_rate", 
    hue="target_emotion", hue_order=EMOTIONS, marker="o", 
    linewidth=2, palette=desaturated_palette, ax=ax1
)
ax1.set_xlabel("Layer Index")
ax1.set_ylabel("Prediction Flip Rate")
ax1.set_title("(A) Layer-wise Ablation Impact", pad=12, fontsize=12, fontweight="normal")
ax1.legend(title="Emotion", fontsize=9, title_fontsize=10, loc='upper left', frameon=False)
ax1.grid(False)

# --- PANEL B: Ablation Heatmap (Reversed) ---
p_b = pivot_b.copy()
p_b.index = [prettify_model_label(x) for x in p_b.index]
p_b.columns = [str(x).capitalize() for x in p_b.columns]
p_b = p_b.reindex(sorted(p_b.index, key=make_sorting_key))

sns.heatmap(
    p_b, ax=ax2, cmap="Blues", annot=True, fmt=".1%", linewidths=1.2, 
    cbar_kws={"fraction": 0.04, "pad": 0.04}
)
cbar_b = ax2.collections[0].colorbar
cbar_b.outline.set_visible(False)
cbar_b.ax.tick_params(size=0)
cbar_b.set_label("Max flip rate", labelpad=2, fontsize=11, fontweight="medium")
ax2.set_title("(B) Ablation Summary", pad=12, fontsize=12, fontweight="normal")

# --- PANEL C: Controllability Heatmap (Reversed) ---
p_c = pivot_c.copy()
p_c.index = [prettify_model_label(x) for x in p_c.index]
p_c.columns = [str(x).capitalize() for x in p_c.columns]
p_c = p_c.reindex(sorted(p_c.index, key=make_sorting_key))

sns.heatmap(
    p_c, ax=ax3, cmap="Blues", annot=True, fmt=".1%", linewidths=1.2, 
    cbar_kws={"fraction": 0.04, "pad": 0.04}
)
cbar_c = ax3.collections[0].colorbar
cbar_c.outline.set_visible(False)
cbar_c.ax.tick_params(size=0)
cbar_c.set_label("Mean flip-to-target", labelpad=2, fontsize=11, fontweight="medium")
ax3.set_title("(C) Activation Patching Summary", pad=12, fontsize=12, fontweight="normal")

# ============================================================
# 4. SHARED AXIS CLEANUP
# ============================================================
for idx, ax in enumerate(axes):
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    ax.set_xlabel("")
    
    if idx > 0:
        ax.set_ylabel("")
        ax.tick_params(which="both", bottom=False, left=False)
        
        if idx == 2:
            plt.setp(ax.get_yticklabels(), visible=False)
            ax.yaxis.set_ticks_position('none')
        else:
            ax.set_yticklabels(ax.get_yticklabels(), fontweight="medium", rotation=0)

composite_path = os.path.join(FINAL_VISUALS_DIR, "composite_report_refined_gap_desat.png")
plt.savefig(composite_path, dpi=300, bbox_inches="tight", facecolor="white")
print("\n" + "="*50)
print("SUCCESS: Refined 3-panel composite with tighter gap and desaturated lines.")
print(f"Saved to: {composite_path}")
print("="*50)

plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np

# ============================================================
# Config
# ============================================================

MODEL_RUNS = [
    ("google/gemma-2-2b", None),
    ("google/gemma-2-9b", None),

    ("google/gemma-3-4b-pt", "big"),
    ("google/gemma-3-4b-pt", "small"),
    ("google/gemma-3-4b-it", "big"),
    ("google/gemma-3-4b-it", "small"),

    ("meta-llama/Llama-3.1-8B", None),

    ("google/gemma-3-12b-pt", "small"),
    ("google/gemma-3-12b-it", "small"),
    ("google/gemma-3-12b-it", "big"),
    ("google/gemma-3-12b-pt", "big"),
]

EMOTIONS = [
    "joy",
    "surprise",
    "disgust",
    "anger",
    "sadness",
    "fear",
]

OUT_DIR = "paper_visuals/feature_statistics"
os.makedirs(OUT_DIR, exist_ok=True)


# ============================================================
# Helpers
# ============================================================

def model_slug(model_name):
    return model_name.replace("/", "-")


def run_name(model_name, sae_size):
    slug = model_slug(model_name)

    if sae_size is None:
        return slug

    return f"{slug}-{sae_size}"


def feature_dir_from_model(model_name, sae_size):
    slug = model_slug(model_name)

    if sae_size is None:
        return f"data/{slug}-feature-act"

    return f"data/{slug}-feature-act-{sae_size}"


def feature_csv_path(model_name, sae_size):
    feature_dir = feature_dir_from_model(model_name, sae_size)

    return os.path.join(
        feature_dir,
        "auto_selected_unique_features_with_affected_emotions.csv",
    )


def parse_emotions(x):
    if pd.isna(x):
        return set()

    return set(
        v.strip().lower()
        for v in str(x).split(",")
        if v.strip()
    )


# ============================================================
# Statistics
# ============================================================

all_model_stats = []
all_emotion_stats = []
all_layer_stats = []
all_overlap_stats = []

for model_name, sae_size in MODEL_RUNS:

    run = run_name(model_name, sae_size)

    csv_path = feature_csv_path(model_name, sae_size)

    print("\n" + "=" * 80)
    print(run)
    print(csv_path)
    print("=" * 80)

    if not os.path.exists(csv_path):
        print("Missing:", csv_path)
        continue

    df = pd.read_csv(csv_path)

    required_cols = [
        "layer_idx",
        "feature_idx",
        "affected_emotions",
    ]

    missing = [c for c in required_cols if c not in df.columns]

    if len(missing) > 0:
        print("Missing columns:", missing)
        continue

    # --------------------------------------------------------
    # Basic cleanup
    # --------------------------------------------------------

    df["layer_idx"] = df["layer_idx"].astype(int)
    df["feature_idx"] = df["feature_idx"].astype(int)

    df["affected_emotion_set"] = df["affected_emotions"].apply(
        parse_emotions
    )

    df["n_affected_emotions"] = df["affected_emotion_set"].apply(len)

    df_unique = df.drop_duplicates(
        subset=["layer_idx", "feature_idx"]
    ).copy()

    # --------------------------------------------------------
    # Global model statistics
    # --------------------------------------------------------

    total_unique_features = len(df_unique)

    total_rows = len(df)

    multi_emotion_features = (
        df_unique["n_affected_emotions"] > 1
    ).sum()

    single_emotion_features = (
        df_unique["n_affected_emotions"] == 1
    ).sum()

    avg_emotions_per_feature = (
        df_unique["n_affected_emotions"].mean()
    )

    max_layer = df_unique["layer_idx"].max()

    min_layer = df_unique["layer_idx"].min()

    all_model_stats.append(
        {
            "model_name": model_name,
            "sae_size": sae_size if sae_size is not None else "none",
            "run": run,

            "n_rows": total_rows,
            "n_unique_features": total_unique_features,

            "n_single_emotion_features": single_emotion_features,
            "n_multi_emotion_features": multi_emotion_features,

            "pct_multi_emotion_features":
                multi_emotion_features / total_unique_features
                if total_unique_features > 0 else np.nan,

            "avg_emotions_per_feature":
                avg_emotions_per_feature,

            "min_layer": min_layer,
            "max_layer": max_layer,
        }
    )

    # --------------------------------------------------------
    # Emotion statistics
    # --------------------------------------------------------

    for emotion in EMOTIONS:

        sub = df_unique[
            df_unique["affected_emotion_set"].apply(
                lambda s: emotion in s
            )
        ].copy()

        n_features = len(sub)

        if n_features > 0:
            avg_layer = sub["layer_idx"].mean()
            median_layer = sub["layer_idx"].median()
            std_layer = sub["layer_idx"].std()

            earliest_layer = sub["layer_idx"].min()
            latest_layer = sub["layer_idx"].max()

        else:
            avg_layer = np.nan
            median_layer = np.nan
            std_layer = np.nan

            earliest_layer = np.nan
            latest_layer = np.nan

        all_emotion_stats.append(
            {
                "model_name": model_name,
                "sae_size": sae_size if sae_size is not None else "none",
                "run": run,

                "emotion": emotion,

                "n_features": n_features,

                "avg_layer": avg_layer,
                "median_layer": median_layer,
                "std_layer": std_layer,

                "earliest_layer": earliest_layer,
                "latest_layer": latest_layer,
            }
        )

    # --------------------------------------------------------
    # Layer statistics
    # --------------------------------------------------------

    layer_counts = (
        df_unique
        .groupby("layer_idx")
        .size()
        .reset_index(name="n_features")
    )

    for _, row in layer_counts.iterrows():

        all_layer_stats.append(
            {
                "model_name": model_name,
                "sae_size": sae_size if sae_size is not None else "none",
                "run": run,

                "layer_idx": int(row["layer_idx"]),
                "n_features": int(row["n_features"]),
            }
        )

    # --------------------------------------------------------
    # Emotion overlap statistics
    # --------------------------------------------------------

    emotion_to_features = {}

    for emotion in EMOTIONS:

        sub = df_unique[
            df_unique["affected_emotion_set"].apply(
                lambda s: emotion in s
            )
        ].copy()

        feature_set = set(
            zip(
                sub["layer_idx"].astype(int),
                sub["feature_idx"].astype(int),
            )
        )

        emotion_to_features[emotion] = feature_set

    for emotion_a in EMOTIONS:
        for emotion_b in EMOTIONS:

            set_a = emotion_to_features[emotion_a]
            set_b = emotion_to_features[emotion_b]

            inter = len(set_a & set_b)
            union = len(set_a | set_b)

            jaccard = (
                inter / union
                if union > 0 else np.nan
            )

            overlap_coef = (
                inter / min(len(set_a), len(set_b))
                if min(len(set_a), len(set_b)) > 0
                else np.nan
            )

            all_overlap_stats.append(
                {
                    "model_name": model_name,
                    "sae_size": sae_size if sae_size is not None else "none",
                    "run": run,

                    "emotion_a": emotion_a,
                    "emotion_b": emotion_b,

                    "intersection": inter,
                    "union": union,

                    "jaccard_overlap": jaccard,
                    "overlap_coefficient": overlap_coef,
                }
            )

# ============================================================
# Save outputs
# ============================================================

model_stats_df = pd.DataFrame(all_model_stats)
emotion_stats_df = pd.DataFrame(all_emotion_stats)
layer_stats_df = pd.DataFrame(all_layer_stats)
overlap_stats_df = pd.DataFrame(all_overlap_stats)

model_stats_path = os.path.join(
    OUT_DIR,
    "model_feature_statistics.csv",
)

emotion_stats_path = os.path.join(
    OUT_DIR,
    "emotion_feature_statistics.csv",
)

layer_stats_path = os.path.join(
    OUT_DIR,
    "layer_feature_statistics.csv",
)

overlap_stats_path = os.path.join(
    OUT_DIR,
    "emotion_overlap_statistics.csv",
)

model_stats_df.to_csv(model_stats_path, index=False)
emotion_stats_df.to_csv(emotion_stats_path, index=False)
layer_stats_df.to_csv(layer_stats_path, index=False)
overlap_stats_df.to_csv(overlap_stats_path, index=False)

print("\n" + "=" * 80)
print("Saved:")
print(model_stats_path)
print(emotion_stats_path)
print(layer_stats_path)
print(overlap_stats_path)
print("=" * 80)

# ============================================================
# Aggregate summaries
# ============================================================

print("\nMODEL SUMMARY")
print(model_stats_df)

print("\nEMOTION SUMMARY (mean across models)")
print(
    emotion_stats_df
    .groupby("emotion")
    .agg(
        mean_n_features=("n_features", "mean"),
        mean_avg_layer=("avg_layer", "mean"),
        mean_latest_layer=("latest_layer", "mean"),
    )
    .round(2)
)

# ============================================================
# Average statistics:
# 1. average # emotions per feature
# 2. average # features per emotion
# 3. average # features per emotion per layer
# ============================================================

# ------------------------------------------------------------
# Average # emotions per feature
# ------------------------------------------------------------

avg_emotions_per_feature_df = (
    model_stats_df[
        [
            "run",
            "avg_emotions_per_feature",
        ]
    ]
    .copy()
)

avg_emotions_per_feature_path = os.path.join(
    OUT_DIR,
    "avg_emotions_per_feature.csv",
)

avg_emotions_per_feature_df.to_csv(
    avg_emotions_per_feature_path,
    index=False,
)

print("\nAverage # emotions per feature")
print(avg_emotions_per_feature_df)

print("\nOverall mean:")
print(
    avg_emotions_per_feature_df[
        "avg_emotions_per_feature"
    ].mean()
)


# ------------------------------------------------------------
# Average # features per emotion
# ------------------------------------------------------------

avg_features_per_emotion_df = (
    emotion_stats_df
    .groupby("emotion")
    .agg(
        mean_n_features=("n_features", "mean"),
        std_n_features=("n_features", "std"),
        min_n_features=("n_features", "min"),
        max_n_features=("n_features", "max"),
    )
    .reset_index()
)

avg_features_per_emotion_path = os.path.join(
    OUT_DIR,
    "avg_features_per_emotion.csv",
)

avg_features_per_emotion_df.to_csv(
    avg_features_per_emotion_path,
    index=False,
)

print("\nAverage # features per emotion")
print(avg_features_per_emotion_df)


# ------------------------------------------------------------
# Average # features per emotion per layer
# ------------------------------------------------------------

all_emotion_layer_rows = []

for model_name, sae_size in MODEL_RUNS:

    run = run_name(model_name, sae_size)

    csv_path = feature_csv_path(model_name, sae_size)

    if not os.path.exists(csv_path):
        continue

    df = pd.read_csv(csv_path)

    df["layer_idx"] = df["layer_idx"].astype(int)

    df["affected_emotion_set"] = df["affected_emotions"].apply(
        parse_emotions
    )

    df_unique = df.drop_duplicates(
        subset=["layer_idx", "feature_idx"]
    ).copy()

    for emotion in EMOTIONS:

        sub = df_unique[
            df_unique["affected_emotion_set"].apply(
                lambda s: emotion in s
            )
        ].copy()

        layer_counts = (
            sub
            .groupby("layer_idx")
            .size()
            .reset_index(name="n_features")
        )

        for _, row in layer_counts.iterrows():

            all_emotion_layer_rows.append(
                {
                    "run": run,
                    "emotion": emotion,
                    "layer_idx": int(row["layer_idx"]),
                    "n_features": int(row["n_features"]),
                }
            )

emotion_layer_df = pd.DataFrame(all_emotion_layer_rows)

avg_features_per_emotion_per_layer_df = (
    emotion_layer_df
    .groupby(["emotion", "layer_idx"])
    .agg(
        mean_n_features=("n_features", "mean"),
        std_n_features=("n_features", "std"),
        n_models=("n_features", "count"),
    )
    .reset_index()
)

avg_features_per_emotion_per_layer_path = os.path.join(
    OUT_DIR,
    "avg_features_per_emotion_per_layer.csv",
)

avg_features_per_emotion_per_layer_df.to_csv(
    avg_features_per_emotion_per_layer_path,
    index=False,
)

print("\nAverage # features per emotion per layer")
print(avg_features_per_emotion_per_layer_df)


# ------------------------------------------------------------
# Overall summary
# ------------------------------------------------------------

summary_rows = []

for emotion in EMOTIONS:

    sub = avg_features_per_emotion_df[
        avg_features_per_emotion_df["emotion"] == emotion
    ]

    if len(sub) == 0:
        continue

    summary_rows.append(
        {
            "emotion": emotion,

            "avg_features_per_emotion":
                float(sub["mean_n_features"].iloc[0]),

            "avg_features_per_layer":
                float(
                    avg_features_per_emotion_per_layer_df[
                        avg_features_per_emotion_per_layer_df["emotion"] == emotion
                    ]["mean_n_features"].mean()
                ),
        }
    )

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(
    OUT_DIR,
    "summary_statistics.csv",
)

summary_df.to_csv(summary_path, index=False)

print("\nSummary")
print(summary_df)

print("\nSaved:")
print(avg_emotions_per_feature_path)
print(avg_features_per_emotion_path)
print(avg_features_per_emotion_per_layer_path)
print(summary_path)

In [ ]:
model_stats_df["n_unique_features"].mean()

In [ ]:
overall_avg = (
    avg_features_per_layer_per_emotion_df[
        "avg_features_per_layer"
    ].mean()
)

print("Overall average # features per layer per emotion:")
print(overall_avg)

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Config
# ============================================================

BASE_DIRS = [
    "data/google-gemma-2-2b-feature-act/manual_weak_emotion_boosting",
    "data/google-gemma-2-9b-feature-act/manual_weak_emotion_boosting",
    "data/google-gemma-3-4b-pt-feature-act/manual_weak_emotion_boosting",
    "data/google-gemma-3-4b-it-feature-act/manual_weak_emotion_boosting",
    "data/google-gemma-3-12b-pt-feature-act/manual_weak_emotion_boosting",
    "data/google-gemma-3-12b-it-feature-act/manual_weak_emotion_boosting",
]

OUTPUT_DIR = "paper_visuals/manual_weak_emotion_boosting"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

MODEL_ORDER = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

MODEL_LABELS = {
    "google/gemma-2-2b": "Gemma-2-2B",
    "google/gemma-2-9b": "Gemma-2-9B",
    "google/gemma-3-4b-pt": "Gemma-3-4B-PT",
    "google/gemma-3-4b-it": "Gemma-3-4B-IT",
    "google/gemma-3-12b-pt": "Gemma-3-12B-PT",
    "google/gemma-3-12b-it": "Gemma-3-12B-IT",
}

MODEL_FROM_DIR = {
    "google-gemma-2-2b-feature-act": "google/gemma-2-2b",
    "google-gemma-2-9b-feature-act": "google/gemma-2-9b",
    "google-gemma-3-4b-pt-feature-act": "google/gemma-3-4b-pt",
    "google-gemma-3-4b-it-feature-act": "google/gemma-3-4b-it",
    "google-gemma-3-12b-pt-feature-act": "google/gemma-3-12b-pt",
    "google-gemma-3-12b-it-feature-act": "google/gemma-3-12b-it",
}

sns.set_theme(style="whitegrid", context="paper")


# ============================================================
# Loading
# ============================================================

def infer_run_idx(path):
    name = os.path.basename(path)
    if "run1" in name:
        return 1
    if "run2" in name:
        return 2
    return None


def infer_model_name(path):
    parts = path.split(os.sep)
    for part in parts:
        if part in MODEL_FROM_DIR:
            return MODEL_FROM_DIR[part]
    return None


def load_all_results_from_model_folders():
    summary_dfs = []
    per_emotion_dfs = []

    for base_dir in BASE_DIRS:
        summary_paths = sorted(
            glob.glob(os.path.join(base_dir, "multiplier_sweep_run*.csv"))
        )

        summary_paths = [
            p for p in summary_paths
            if not p.endswith("_per_emotion.csv")
            and not p.endswith("_all_predictions.csv")
        ]

        per_emotion_paths = sorted(
            glob.glob(os.path.join(base_dir, "multiplier_sweep_run*_per_emotion.csv"))
        )

        for path in summary_paths:
            df = pd.read_csv(path)
            df["run_idx"] = infer_run_idx(path)
            df["model_name"] = infer_model_name(path)
            df["source_file"] = path
            summary_dfs.append(df)

        for path in per_emotion_paths:
            df = pd.read_csv(path)
            df["run_idx"] = infer_run_idx(path)
            df["model_name"] = infer_model_name(path)
            df["source_file"] = path
            per_emotion_dfs.append(df)

    if not summary_dfs:
        raise FileNotFoundError("No summary files found.")

    if not per_emotion_dfs:
        raise FileNotFoundError("No per-emotion files found.")

    summary_df = pd.concat(summary_dfs, ignore_index=True)
    per_emotion_df = pd.concat(per_emotion_dfs, ignore_index=True)

    summary_df = summary_df.dropna(subset=["run_idx", "model_name"])
    per_emotion_df = per_emotion_df.dropna(subset=["run_idx", "model_name"])

    return summary_df, per_emotion_df


summary_df, per_emotion_df = load_all_results_from_model_folders()


# ============================================================
# Standardize columns
# ============================================================

summary_df["run_idx"] = summary_df["run_idx"].astype(int)
per_emotion_df["run_idx"] = per_emotion_df["run_idx"].astype(int)

summary_df["multiplier"] = pd.to_numeric(summary_df["multiplier"], errors="coerce")
per_emotion_df["multiplier"] = pd.to_numeric(per_emotion_df["multiplier"], errors="coerce")

summary_df["model_label"] = summary_df["model_name"].map(MODEL_LABELS)
per_emotion_df["model_label"] = per_emotion_df["model_name"].map(MODEL_LABELS)

summary_df["model_label"] = pd.Categorical(
    summary_df["model_label"],
    categories=[MODEL_LABELS[m] for m in MODEL_ORDER],
    ordered=True,
)

per_emotion_df["model_label"] = pd.Categorical(
    per_emotion_df["model_label"],
    categories=[MODEL_LABELS[m] for m in MODEL_ORDER],
    ordered=True,
)

summary_df = summary_df.sort_values(["run_idx", "model_label", "multiplier"])
per_emotion_df = per_emotion_df.sort_values(
    ["run_idx", "model_label", "emotion", "multiplier"]
)

print("Loaded summary results:")
print(summary_df[["model_name", "run_idx", "multiplier", "after_macro_f1", "changed_prediction_rate"]])

print("\nLoaded per-emotion results:")
print(per_emotion_df[["model_name", "run_idx", "emotion", "multiplier", "f1_after"]].head())


# ============================================================
# Plot helpers
# ============================================================

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def plot_macro_f1(summary_df, run_idx):
    df = summary_df[summary_df["run_idx"] == run_idx].copy()

    if df.empty:
        print(f"No data for run {run_idx}, macro-F1 skipped.")
        return

    plt.figure(figsize=(7.5, 4.6))

    sns.lineplot(
        data=df,
        x="multiplier",
        y="after_macro_f1",
        hue="model_label",
        hue_order=[MODEL_LABELS[m] for m in MODEL_ORDER],
        marker="o",
        linewidth=2,
        errorbar=None,
    )

    plt.xlabel("Multiplier k")
    plt.ylabel("Macro-F1")
    plt.title(f"Run {run_idx}: Macro-F1 by k")
    plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    savefig(os.path.join(OUTPUT_DIR, f"run{run_idx}_macro_f1_by_k.png"))


def plot_flip_rate(summary_df, run_idx):
    df = summary_df[summary_df["run_idx"] == run_idx].copy()

    if df.empty:
        print(f"No data for run {run_idx}, flip-rate skipped.")
        return

    plt.figure(figsize=(7.5, 4.6))

    sns.lineplot(
        data=df,
        x="multiplier",
        y="changed_prediction_rate",
        hue="model_label",
        hue_order=[MODEL_LABELS[m] for m in MODEL_ORDER],
        marker="o",
        linewidth=2,
        errorbar=None,
    )

    plt.xlabel("Multiplier k")
    plt.ylabel("Flip rate")
    plt.title(f"Run {run_idx}: Flip rate by k")
    plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    savefig(os.path.join(OUTPUT_DIR, f"run{run_idx}_flip_rate_by_k.png"))


def plot_per_emotion_f1_for_each_model(per_emotion_df, run_idx):
    df_run = per_emotion_df[per_emotion_df["run_idx"] == run_idx].copy()

    if df_run.empty:
        print(f"No data for run {run_idx}, per-emotion F1 skipped.")
        return

    for model_name in MODEL_ORDER:
        model_label = MODEL_LABELS[model_name]
        df_model = df_run[df_run["model_name"] == model_name].copy()

        if df_model.empty:
            print(f"No data for run {run_idx}, {model_label}, skipped.")
            continue

        plt.figure(figsize=(6.8, 4.5))

        sns.lineplot(
            data=df_model,
            x="multiplier",
            y="f1_after",
            hue="emotion",
            hue_order=EMOTIONS,
            marker="o",
            linewidth=2,
            errorbar=None,
        )

        plt.xlabel("Multiplier k")
        plt.ylabel("Per-emotion F1")
        plt.title(f"Run {run_idx}: {model_label}")
        plt.legend(title="Emotion", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

        safe_model = model_name.replace("/", "-")
        savefig(
            os.path.join(
                OUTPUT_DIR,
                f"run{run_idx}_{safe_model}_per_emotion_f1_by_k.png",
            )
        )


# ============================================================
# Run plots
# ============================================================

for run_idx in [1, 2]:
    plot_macro_f1(summary_df, run_idx)
    plot_flip_rate(summary_df, run_idx)
    plot_per_emotion_f1_for_each_model(per_emotion_df, run_idx)

print(f"\nSaved plots to: {OUTPUT_DIR}")

In [ ]:
import os
import glob
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Config
# ============================================================

BASE_DIRS = [
    "data/google-gemma-2-2b-feature-act/global_feature_steering",
    "data/google-gemma-2-9b-feature-act/global_feature_steering",
    "data/google-gemma-3-4b-pt-feature-act-small/global_feature_steering",
    "data/google-gemma-3-4b-it-feature-act-small/global_feature_steering",
    "data/google-gemma-3-12b-pt-feature-act-small/global_feature_steering",
    "data/google-gemma-3-12b-it-feature-act-small/global_feature_steering",
]

OUTPUT_DIR = "paper_visuals/global_feature_steering_multiplier_analysis"
TABLE_DIR = "paper_tables/global_feature_steering_multiplier_analysis"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

MODEL_LABELS = {
    "google-gemma-2-2b-feature-act": "Gemma-2-2B",
    "google-gemma-2-9b-feature-act": "Gemma-2-9B",
    "google-gemma-3-4b-pt-feature-act-small": "Gemma-3-4B-PT",
    "google-gemma-3-4b-it-feature-act-small": "Gemma-3-4B-IT",
    "google-gemma-3-12b-pt-feature-act-small": "Gemma-3-12B-PT",
    "google-gemma-3-12b-it-feature-act-small": "Gemma-3-12B-IT",
}

sns.set_theme(style="whitegrid", context="paper")


# ============================================================
# Helpers
# ============================================================

def savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()


def parse_feature_emotions(x):
    """
    Parse feature emotion field.
    Supports:
    - stringified list
    - comma separated
    - single label
    """

    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    x = str(x).strip()

    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return [str(v).strip().lower() for v in parsed]
    except Exception:
        pass

    if "," in x:
        return [v.strip().lower() for v in x.split(",")]

    return [x.lower()]


# ============================================================
# Load learned multipliers
# ============================================================

all_rows = []

for base_dir in BASE_DIRS:

    run_dirs = sorted(glob.glob(os.path.join(base_dir, "lr_*")))

    for run_dir in run_dirs:

        mult_path = os.path.join(run_dir, "learned_feature_multipliers.csv")

        if not os.path.exists(mult_path):
            print(f"[Missing] {mult_path}")
            continue

        df = pd.read_csv(mult_path)

        model_key = None

        for k in MODEL_LABELS:
            if k in run_dir:
                model_key = k
                break

        df["model_key"] = model_key
        df["model_label"] = MODEL_LABELS.get(model_key, model_key)
        df["run_dir"] = run_dir

        # ----------------------------------------------------
        # Infer multiplier column
        # ----------------------------------------------------

        multiplier_col = None

        for c in [
            "multiplier",
            "learned_multiplier",
            "feature_multiplier",
            "value",
        ]:
            if c in df.columns:
                multiplier_col = c
                break

        if multiplier_col is None:
            raise ValueError(
                f"Cannot find multiplier column in {mult_path}\nColumns: {list(df.columns)}"
            )

        df["multiplier_value"] = pd.to_numeric(df[multiplier_col], errors="coerce")

        # ----------------------------------------------------
        # Infer emotion ownership
        # ----------------------------------------------------

        emotion_col = None

        for c in [
            "affected_emotions",
            "emotion",
            "emotions",
            "target_emotions",
        ]:
            if c in df.columns:
                emotion_col = c
                break

        if emotion_col is None:
            raise ValueError(
                f"Cannot find emotion column in {mult_path}\nColumns: {list(df.columns)}"
            )

        df["emotion_list"] = df[emotion_col].apply(parse_feature_emotions)

        exploded = df.explode("emotion_list").rename(
            columns={"emotion_list": "emotion"}
        )

        exploded["emotion"] = exploded["emotion"].astype(str).str.lower()

        exploded = exploded[exploded["emotion"].isin(EMOTIONS)]

        all_rows.append(exploded)

if not all_rows:
    raise ValueError("No learned_feature_multipliers.csv files found.")

mult_df = pd.concat(all_rows, ignore_index=True)

print(mult_df.head())


# ============================================================
# Basic statistics
# ============================================================

summary_stats = (
    mult_df.groupby(["model_label", "emotion"])
    .agg(
        mean_multiplier=("multiplier_value", "mean"),
        median_multiplier=("multiplier_value", "median"),
        std_multiplier=("multiplier_value", "std"),
        n_features=("multiplier_value", "size"),
        pct_gt1=("multiplier_value", lambda x: (x > 1.0).mean()),
        pct_lt1=("multiplier_value", lambda x: (x < 1.0).mean()),
    )
    .reset_index()
)

summary_stats.to_csv(
    os.path.join(TABLE_DIR, "multiplier_summary_stats.csv"),
    index=False,
)

print(summary_stats)


# ============================================================
# Detailed analysis plots
# ============================================================

# ------------------------------------------------------------
# Distribution by emotion
# ------------------------------------------------------------

plt.figure(figsize=(8.2, 4.8))

sns.violinplot(
    data=mult_df,
    x="emotion",
    y="multiplier_value",
    inner="box",
    cut=0,
)

plt.axhline(1.0, linestyle="--", alpha=0.5)

plt.xlabel("Feature emotion")
plt.ylabel("Learned multiplier")
plt.title("Distribution of learned multipliers by emotion")

savefig(
    os.path.join(
        OUTPUT_DIR,
        "multiplier_distribution_by_emotion.png",
    )
)


# ------------------------------------------------------------
# Mean multiplier per emotion across models
# ------------------------------------------------------------

agg = (
    mult_df.groupby(["model_label", "emotion"])
    .agg(
        mean_multiplier=("multiplier_value", "mean"),
        std_multiplier=("multiplier_value", "std"),
    )
    .reset_index()
)

plt.figure(figsize=(8.5, 4.8))

sns.barplot(
    data=agg,
    x="emotion",
    y="mean_multiplier",
    hue="model_label",
)

plt.axhline(1.0, linestyle="--", alpha=0.5)

plt.xlabel("Feature emotion")
plt.ylabel("Mean learned multiplier")
plt.title("Average learned multiplier by emotion")

plt.legend(
    title="Model",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)

savefig(
    os.path.join(
        OUTPUT_DIR,
        "mean_multiplier_by_emotion_and_model.png",
    )
)


# ------------------------------------------------------------
# Amplified vs suppressed feature ratios
# ------------------------------------------------------------

ratio_df = (
    mult_df.groupby(["model_label", "emotion"])
    .agg(
        amplified_ratio=("multiplier_value", lambda x: (x > 1.0).mean()),
        suppressed_ratio=("multiplier_value", lambda x: (x < 1.0).mean()),
    )
    .reset_index()
)

ratio_long = ratio_df.melt(
    id_vars=["model_label", "emotion"],
    value_vars=["amplified_ratio", "suppressed_ratio"],
    var_name="direction",
    value_name="ratio",
)

plt.figure(figsize=(9.2, 4.8))

sns.barplot(
    data=ratio_long,
    x="emotion",
    y="ratio",
    hue="direction",
)

plt.xlabel("Feature emotion")
plt.ylabel("Feature ratio")
plt.title("Amplified vs. suppressed feature ratios")

savefig(
    os.path.join(
        OUTPUT_DIR,
        "amplified_vs_suppressed_feature_ratios.png",
    )
)


# ============================================================
# Comprehensive analysis
# Which emotion features are amplified/suppressed?
# ============================================================

emotion_global = (
    mult_df.groupby("emotion")
    .agg(
        mean_multiplier=("multiplier_value", "mean"),
        median_multiplier=("multiplier_value", "median"),
        amplified_ratio=("multiplier_value", lambda x: (x > 1.0).mean()),
        suppressed_ratio=("multiplier_value", lambda x: (x < 1.0).mean()),
        n_features=("multiplier_value", "size"),
    )
    .reset_index()
)

emotion_global["net_direction"] = (
    emotion_global["amplified_ratio"]
    - emotion_global["suppressed_ratio"]
)

emotion_global = emotion_global.sort_values(
    "mean_multiplier",
    ascending=False,
)

emotion_global.to_csv(
    os.path.join(
        TABLE_DIR,
        "global_emotion_multiplier_summary.csv",
    ),
    index=False,
)

print("\nGlobal emotion multiplier summary:")
print(emotion_global)


# ------------------------------------------------------------
# Global emotion tendency plot
# ------------------------------------------------------------

plt.figure(figsize=(7.5, 4.8))

sns.barplot(
    data=emotion_global,
    x="emotion",
    y="mean_multiplier",
)

plt.axhline(1.0, linestyle="--", alpha=0.5)

plt.xlabel("Feature emotion")
plt.ylabel("Mean learned multiplier")
plt.title("Global amplification tendency by emotion")

savefig(
    os.path.join(
        OUTPUT_DIR,
        "global_amplification_tendency_by_emotion.png",
    )
)


# ------------------------------------------------------------
# Net amplification score
# ------------------------------------------------------------

plt.figure(figsize=(7.5, 4.8))

sns.barplot(
    data=emotion_global,
    x="emotion",
    y="net_direction",
)

plt.axhline(0.0, linestyle="--", alpha=0.5)

plt.xlabel("Feature emotion")
plt.ylabel("Amplified ratio − suppressed ratio")
plt.title("Net amplification tendency by emotion")

savefig(
    os.path.join(
        OUTPUT_DIR,
        "net_amplification_tendency_by_emotion.png",
    )
)


# ============================================================
# Save comprehensive table
# ============================================================

emotion_global.to_csv(
    os.path.join(
        TABLE_DIR,
        "emotion_feature_amplification_analysis.csv",
    ),
    index=False,
)

latex_df = emotion_global.copy()

for col in [
    "mean_multiplier",
    "median_multiplier",
    "amplified_ratio",
    "suppressed_ratio",
    "net_direction",
]:
    latex_df[col] = latex_df[col].map(lambda x: f"{x:.3f}")

with open(
    os.path.join(
        TABLE_DIR,
        "emotion_feature_amplification_analysis.tex",
    ),
    "w",
) as f:
    f.write(
        latex_df.to_latex(
            index=False,
            escape=False,
        )
    )

print("\nSaved plots to:", OUTPUT_DIR)
print("Saved tables to:", TABLE_DIR)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# ============================================================
# Global Style
# ============================================================

EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
})

sns.set_style("whitegrid")

# ============================================================
# Config
# ============================================================

RUN_LABELS = {
    1: "Enhancing Disgust Features",
    2: "Enhancing Surprise Features",
}

emotion_order = [e.capitalize() for e in EMOTIONS]

# ============================================================
# Helper
# ============================================================

def aggregate_mean_std(df, value_col):
    agg = (
        df.groupby(["run_idx", "emotion", "multiplier"], as_index=False)
        .agg(
            mean_value=(value_col, "mean"),
            std_value=(value_col, "std"),
            n_models=("model_name", "nunique"),
        )
    )

    agg["std_value"] = agg["std_value"].fillna(0.0)
    return agg


# ============================================================
# Layout
# ============================================================

fig = plt.figure(figsize=(12, 3))

gs = fig.add_gridspec(
    1,
    3,
    width_ratios=[0.25, 0.25, 0.5],
    wspace=0.28,
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1], sharey=axA)
axC = fig.add_subplot(gs[0, 2])

# ============================================================
# (A) and (B): Aggregate per-emotion F1
# ============================================================

agg = aggregate_mean_std(per_emotion_df, value_col="f1_after")

for ax, run_idx, panel_label in zip(
    [axA, axB],
    [1, 2],
    ["(A)", "(B)"],
):

    df_run = agg[agg["run_idx"] == run_idx].copy()

    for emotion in EMOTIONS:
        sub = df_run[df_run["emotion"] == emotion].sort_values("multiplier")

        if sub.empty:
            continue

        x = sub["multiplier"].to_numpy()
        mean = sub["mean_value"].to_numpy()
        std = sub["std_value"].to_numpy()

        lower = np.clip(mean - std, 0, 1)
        upper = np.clip(mean + std, 0, 1)

        ax.plot(
            x,
            mean,
            marker="o",
            linewidth=2,
            label=emotion,
        )

        ax.fill_between(
            x,
            lower,
            upper,
            alpha=0.16,
        )

    ax.set_title(
        f"{panel_label} {RUN_LABELS[run_idx]}",
        fontsize=11,
        pad=8,
    )

    ax.set_xlabel(r"Multiplier $\alpha$")
    ax.set_ylim(0, 1.02)

    ax.grid(True, alpha=0.3)

axA.set_ylabel("Mean per-emotion F1")

# Remove duplicated y ticks on B
plt.setp(axB.get_yticklabels(), visible=False)

# Legend between A and B
handles, labels = axA.get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    #title="Emotion",
    loc="center",
    bbox_to_anchor=(0.318, 0.66),
    frameon=False,
    fontsize=8,
    #title_fontsize=9,
)

# ============================================================
# (C): Multiplier distribution violin plot
# ============================================================

plot_df = mult_df.copy()

plot_df["multiplier_plot"] = plot_df["multiplier_value"].clip(0, 10)
plot_df["emotion_label"] = plot_df["emotion"].str.capitalize()

# Violin
sns.violinplot(
    data=plot_df,
    x="emotion_label",
    y="multiplier_plot",
    order=emotion_order,
    color="#5B84B1",
    inner=None,
    linewidth=1.1,
    alpha=0.6,
    cut=0,
    ax=axC,
)

# Mean markers
means = (
    plot_df.groupby("emotion_label")["multiplier_plot"]
    .mean()
    .reindex(emotion_order)
)

axC.scatter(
    x=range(len(emotion_order)),
    y=means,
    color="white",
    edgecolor="black",
    s=36,
    zorder=4,
)

# Boxplot overlay
sns.boxplot(
    data=plot_df,
    x="emotion_label",
    y="multiplier_plot",
    order=emotion_order,
    width=0.18,
    showcaps=True,
    boxprops={
        "facecolor": "white",
        "zorder": 3,
        "alpha": 0.85,
    },
    showfliers=False,
    whiskerprops={"linewidth": 1},
    ax=axC,
)

# Jitter points
sns.stripplot(
    data=plot_df.sample(min(len(plot_df), 3000), random_state=42),
    x="emotion_label",
    y="multiplier_plot",
    order=emotion_order,
    color="black",
    alpha=0.08,
    size=2,
    jitter=0.25,
    ax=axC,
)

axC.axhline(
    1.0,
    linestyle="--",
    alpha=0.6,
    color="black",
)

axC.set_ylim(0, 2)

axC.set_title(
    "(C) Learned multiplier distributions",
    fontsize=11,
    pad=8,
)

axC.set_xlabel("")
axC.set_ylabel("Learned multiplier")

axC.tick_params(axis="x", rotation=30)

# Clean spines
for ax in [axA, axB, axC]:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# ============================================================
# Save
# ============================================================

# os.makedirs("paper_visuals", exist_ok=True)

plt.tight_layout(
    rect=[0, 0.02, 1, 1]
)

plt.savefig(
    "paper_visuals/figure3.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from sae_emotion.visual_utils import (
    run_emotion_overlap_matrix_from_affected,
    find_logit_cols,
)

# ============================================================
# Config
# ============================================================

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

MODELS = [
    {
        "name": "Gemma-2-2B",
        "input_path": "data/google-gemma-2-2b-feature-act/auto_selected_unique_features_with_affected_emotions.csv",
    },
    {
        "name": "Gemma-2-9B",
        "input_path": "data/google-gemma-2-9b-feature-act/auto_selected_unique_features_with_affected_emotions.csv",
    },
    {
        "name": "Gemma-3-4B-PT-big",
        "input_path": "data/google-gemma-3-4b-pt-feature-act-big/auto_selected_unique_features_with_affected_emotions.csv",
    },
    {
        "name": "Gemma-3-4B-PT-small",
        "input_path": "data/google-gemma-3-4b-pt-feature-act-small/auto_selected_unique_features_with_affected_emotions.csv",
    },
    {
        "name": "Gemma-3-4B-IT-big",
        "input_path": "data/google-gemma-3-4b-it-feature-act-big/auto_selected_unique_features_with_affected_emotions.csv",
    },
    {
        "name": "Gemma-3-4B-IT-small",
        "input_path": "data/google-gemma-3-4b-it-feature-act-small/auto_selected_unique_features_with_affected_emotions.csv",
    },
    {
        "name": "Gemma-3-12B-PT-big",
        "input_path": "data/google-gemma-3-12b-pt-feature-act-big/auto_selected_unique_features_with_affected_emotions.csv",
    },
    {
        "name": "Gemma-3-12B-PT-small",
        "input_path": "data/google-gemma-3-12b-pt-feature-act-small/auto_selected_unique_features_with_affected_emotions.csv",
    },
    {
        "name": "Gemma-3-12B-IT-big",
        "input_path": "data/google-gemma-3-12b-it-feature-act-big/auto_selected_unique_features_with_affected_emotions.csv",
    },
    {
        "name": "Gemma-3-12B-IT-small",
        "input_path": "data/google-gemma-3-12b-it-feature-act-small/auto_selected_unique_features_with_affected_emotions.csv",
    },
    {
        "name": "Llama-3.1-8B",
        "input_path": "data/meta-llama-Llama-3.1-8B-feature-act/auto_selected_unique_features_with_affected_emotions.csv",
    },
]

OUTPUT_DIR = "paper_visuals/all_models_overlap_subplots"
OUTPUT_PATH_PNG = os.path.join(OUTPUT_DIR, "all_models_jaccard_overlap_subplots.png")
OUTPUT_PATH_PDF = os.path.join(OUTPUT_DIR, "all_models_jaccard_overlap_subplots.pdf")

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# Style
# ============================================================

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# ============================================================
# Load all matrices first
# ============================================================

matrices = []
valid_models = []

for model_cfg in MODELS:
    name = model_cfg["name"]
    input_path = model_cfg["input_path"]

    if not os.path.exists(input_path):
        print(f"[Missing] {name}: {input_path}")
        continue

    print("=" * 80)
    print(name)
    print("=" * 80)

    df = pd.read_csv(input_path)

    # Optional validation
    _ = find_logit_cols(df, emotions=EMOTIONS)

    tmp_output_dir = os.path.join(OUTPUT_DIR, f"_tmp_{name}")
    os.makedirs(tmp_output_dir, exist_ok=True)

    overlap_df, jaccard_mat, overlap_coef_mat, count_mat = (
        run_emotion_overlap_matrix_from_affected(
            df,
            emotions=EMOTIONS,
            output_dir=tmp_output_dir,
        )
    )

    display_df = (
        jaccard_mat
        .reindex(index=EMOTIONS, columns=EMOTIONS)
        .copy()
    )

    display_df.index = [str(x).capitalize() for x in display_df.index]
    display_df.columns = [str(x).capitalize() for x in display_df.columns]

    matrices.append(display_df)
    valid_models.append(name)


print(f"\nLoaded {len(matrices)} valid model matrices.")


# ============================================================
# Plot all models as subplots
# ============================================================

n_models = len(matrices)

# Good layout for 11 models: 3 rows x 4 cols, last panel empty
ncols = 4
nrows = int(np.ceil(n_models / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(15.5, 10.5),
)

axes = np.array(axes).reshape(-1)

# Shared mask
mask = np.eye(len(EMOTIONS), dtype=bool)

# Same segmented contrast scale as your original code
boundaries = [0.0, 0.02, 0.04, 0.06, 0.09, 0.12, 0.16, 0.20, 0.25, 0.30]

base_cmap = plt.get_cmap("Blues")
custom_colors = base_cmap(np.linspace(0.05, 0.95, len(boundaries) - 1))
segmented_cmap = mpl.colors.ListedColormap(custom_colors)
segmented_cmap.set_bad(color="#f4f4f4")
segmented_norm = mpl.colors.BoundaryNorm(boundaries, segmented_cmap.N)

# One shared colorbar axis
cbar_ax = fig.add_axes([0.92, 0.20, 0.012, 0.60])

for idx, (ax, display_df, model_name) in enumerate(zip(axes, matrices, valid_models)):
    show_cbar = idx == 0

    sns.heatmap(
        display_df,
        ax=ax,
        mask=mask,
        cmap=segmented_cmap,
        norm=segmented_norm,
        annot=True,
        fmt=".2f",
        linewidths=0.8,
        linecolor="white",
        square=True,
        cbar=show_cbar,
        cbar_ax=cbar_ax if show_cbar else None,
        annot_kws={"fontsize": 6.8},
    )

    ax.set_title(model_name, fontsize=9.5, fontweight="medium", pad=4)

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=35,
        ha="right",
        rotation_mode="anchor",
    )
    ax.set_yticklabels(
        ax.get_yticklabels(),
        rotation=0,
    )

    ax.tick_params(which="both", bottom=False, left=False, length=0)

# Hide unused axes
for j in range(n_models, len(axes)):
    axes[j].axis("off")

# Clean shared colorbar
cbar = axes[0].collections[0].colorbar
cbar.outline.set_visible(False)
cbar.ax.tick_params(size=0, labelsize=8)
cbar.set_ticks([boundaries[0], boundaries[-1]])
cbar.ax.set_yticklabels([f"{boundaries[0]:.2f}", f"{boundaries[-1]:.2f}"])
cbar.set_label("Jaccard Overlap Index", labelpad=9, fontsize=9)

# Layout
fig.subplots_adjust(
    left=0.04,
    right=0.90,
    top=0.94,
    bottom=0.06,
    wspace=0.18,
    hspace=0.35,
)

plt.savefig(OUTPUT_PATH_PNG, dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(OUTPUT_PATH_PDF, bbox_inches="tight", facecolor="white")

print("Saved figure:", OUTPUT_PATH_PNG)
print("Saved figure:", OUTPUT_PATH_PDF)

plt.show()

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns


# ============================================================
# Config
# ============================================================

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

MODEL_RUNS = [
    {
        "name": "Gemma-2-2B",
        "safe_name": "google-gemma-2-2b",
    },
    {
        "name": "Gemma-2-9B",
        "safe_name": "google-gemma-2-9b",
    },
    {
        "name": "Gemma-3-4B-PT-big",
        "safe_name": "google-gemma-3-4b-pt-big",
    },
    {
        "name": "Gemma-3-4B-PT-small",
        "safe_name": "google-gemma-3-4b-pt-small",
    },
    {
        "name": "Gemma-3-4B-IT-big",
        "safe_name": "google-gemma-3-4b-it-big",
    },
    {
        "name": "Gemma-3-4B-IT-small",
        "safe_name": "google-gemma-3-4b-it-small",
    },
    {
        "name": "Gemma-3-12B-PT-big",
        "safe_name": "google-gemma-3-12b-pt-big",
    },
    {
        "name": "Gemma-3-12B-PT-small",
        "safe_name": "google-gemma-3-12b-pt-small",
    },
    {
        "name": "Gemma-3-12B-IT-big",
        "safe_name": "google-gemma-3-12b-it-big",
    },
    {
        "name": "Gemma-3-12B-IT-small",
        "safe_name": "google-gemma-3-12b-it-small",
    },
    {
        "name": "Llama-3.1-8B",
        "safe_name": "meta-llama-Llama-3.1-8B",
    },
]

SEARCH_ROOTS = [
    "paper_visuals",
    "data",
]

OUTPUT_DIR = "paper_visuals/all_models_activation_strength_corr_subplots"
OUTPUT_PATH_PNG = os.path.join(
    OUTPUT_DIR,
    "all_models_activation_strength_correlation_subplots.png",
)
OUTPUT_PATH_PDF = os.path.join(
    OUTPUT_DIR,
    "all_models_activation_strength_correlation_subplots.pdf",
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# Helpers
# ============================================================

def find_corr_matrix_path(safe_name: str):
    """
    Find the correct per-model activation-strength correlation matrix.
    Avoids loading aggregate files or unrelated csvs.
    """

    candidates = []

    for root in SEARCH_ROOTS:
        patterns = [
            os.path.join(
                root,
                "**",
                f"*{safe_name}*",
                "emotion_ensemble_strength_correlation_matrix.csv",
            ),
            os.path.join(
                root,
                "**",
                f"*{safe_name}*emotion_ensemble_strength_correlation_matrix.csv",
            ),
            os.path.join(
                root,
                "**",
                f"{safe_name}_emotion_ensemble_strength_correlation_matrix.csv",
            ),
        ]

        for pattern in patterns:
            candidates.extend(glob.glob(pattern, recursive=True))

    candidates = sorted(set(candidates))

    # Remove aggregate-across-model files
    candidates = [
        p for p in candidates
        if "aggregate" not in os.path.basename(p).lower()
    ]

    if len(candidates) == 0:
        return None

    print("Candidate paths:")
    for p in candidates:
        print("  ", p)

    return candidates[0]


def load_corr_matrix(path: str, emotions=EMOTIONS):
    corr_mat = pd.read_csv(path, index_col=0)

    # Normalize labels
    corr_mat.index = corr_mat.index.astype(str).str.lower().str.strip()
    corr_mat.columns = corr_mat.columns.astype(str).str.lower().str.strip()

    # Drop duplicate row/column labels if any
    corr_mat = corr_mat.loc[~corr_mat.index.duplicated(keep="first")]
    corr_mat = corr_mat.loc[:, ~corr_mat.columns.duplicated(keep="first")]

    # Keep only emotion rows/cols
    missing_rows = [e for e in emotions if e not in corr_mat.index]
    missing_cols = [e for e in emotions if e not in corr_mat.columns]

    if missing_rows or missing_cols:
        raise ValueError(
            f"File does not look like a valid emotion correlation matrix:\n"
            f"{path}\n"
            f"Missing rows: {missing_rows}\n"
            f"Missing cols: {missing_cols}\n"
            f"Rows found: {corr_mat.index.tolist()}\n"
            f"Cols found: {corr_mat.columns.tolist()}"
        )

    corr_mat = corr_mat.loc[emotions, emotions].copy()

    display_df = corr_mat.copy()
    display_df.index = [x.capitalize() for x in display_df.index]
    display_df.columns = [x.capitalize() for x in display_df.columns]

    return display_df


# ============================================================
# Style
# ============================================================

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# ============================================================
# Load matrices
# ============================================================

matrices = []
valid_model_names = []
loaded_paths = []

for model_cfg in MODEL_RUNS:
    model_name = model_cfg["name"]
    safe_name = model_cfg["safe_name"]

    corr_path = find_corr_matrix_path(safe_name)

    if corr_path is None:
        print(f"[Missing] {model_name}: could not find correlation matrix.")
        continue

    print("=" * 80)
    print(model_name)
    print("Loaded:", corr_path)
    print("=" * 80)

    display_df = load_corr_matrix(corr_path, emotions=EMOTIONS)

    matrices.append(display_df)
    valid_model_names.append(model_name)
    loaded_paths.append(corr_path)

print(f"\nLoaded {len(matrices)} valid correlation matrices.")


# ============================================================
# Plot all models as subplots
# ============================================================

n_models = len(matrices)

if n_models == 0:
    raise FileNotFoundError(
        "No correlation matrices found. Check SEARCH_ROOTS or file naming."
    )

ncols = 4
nrows = int(np.ceil(n_models / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(15.5, 10.5),
)

axes = np.array(axes).reshape(-1)

mask = np.eye(len(EMOTIONS), dtype=bool)

cmap = plt.get_cmap("RdBu_r").copy()
cmap.set_bad(color="#f4f4f4")

# One shared colorbar axis
cbar_ax = fig.add_axes([0.92, 0.20, 0.012, 0.60])

for idx, (ax, display_df, model_name) in enumerate(
    zip(axes, matrices, valid_model_names)
):
    show_cbar = idx == 0

    sns.heatmap(
        display_df,
        ax=ax,
        mask=mask,
        cmap=cmap,
        annot=True,
        fmt=".2f",
        linewidths=0.8,
        linecolor="white",
        square=True,
        vmin=-1.0,
        vmax=1.0,
        cbar=show_cbar,
        cbar_ax=cbar_ax if show_cbar else None,
        annot_kws={"fontsize": 6.8},
    )

    ax.set_title(model_name, fontsize=9.5, fontweight="medium", pad=4)

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=35,
        ha="right",
        rotation_mode="anchor",
    )

    ax.set_yticklabels(
        ax.get_yticklabels(),
        rotation=0,
    )

    ax.tick_params(which="both", bottom=False, left=False, length=0)

# Hide unused subplots
for j in range(n_models, len(axes)):
    axes[j].axis("off")

# Shared colorbar cleanup
cbar = axes[0].collections[0].colorbar
cbar.outline.set_visible(False)
cbar.ax.tick_params(size=0, labelsize=8)
cbar.set_ticks([-1.0, 0.0, 1.0])
cbar.ax.set_yticklabels(["-1.00", "0.00", "1.00"])
cbar.set_label("Pearson correlation", labelpad=9, fontsize=9)

fig.subplots_adjust(
    left=0.04,
    right=0.90,
    top=0.94,
    bottom=0.06,
    wspace=0.18,
    hspace=0.35,
)

plt.savefig(OUTPUT_PATH_PNG, dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(OUTPUT_PATH_PDF, bbox_inches="tight", facecolor="white")

print("Saved figure:", OUTPUT_PATH_PNG)
print("Saved figure:", OUTPUT_PATH_PDF)

plt.show()


# ============================================================
# Optional: save loaded path list for checking
# ============================================================

path_df = pd.DataFrame({
    "model": valid_model_names,
    "corr_matrix_path": loaded_paths,
})

path_df.to_csv(
    os.path.join(OUTPUT_DIR, "loaded_correlation_matrix_paths.csv"),
    index=False,
)

display(path_df)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import numpy as np


# ============================================================
# Config
# ============================================================

EMOTIONS = [
    "joy",
    "surprise",
    "disgust",
    "anger",
    "sadness",
    "fear",
]

MODEL_RUNS = [
    {
        "name": "Gemma-2-2B",
        "csv_path": (
            "data/google-gemma-2-2b-feature-act/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        ),
    },
    {
        "name": "Gemma-2-9B",
        "csv_path": (
            "data/google-gemma-2-9b-feature-act/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        ),
    },
    {
        "name": "Gemma-3-4B-PT-big",
        "csv_path": (
            "data/google-gemma-3-4b-pt-feature-act-big/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        ),
    },
    {
        "name": "Gemma-3-4B-PT-small",
        "csv_path": (
            "data/google-gemma-3-4b-pt-feature-act-small/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        ),
    },
    {
        "name": "Gemma-3-4B-IT-big",
        "csv_path": (
            "data/google-gemma-3-4b-it-feature-act-big/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        ),
    },
    {
        "name": "Gemma-3-4B-IT-small",
        "csv_path": (
            "data/google-gemma-3-4b-it-feature-act-small/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        ),
    },
    {
        "name": "Gemma-3-12B-PT-big",
        "csv_path": (
            "data/google-gemma-3-12b-pt-feature-act-big/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        ),
    },
    {
        "name": "Gemma-3-12B-PT-small",
        "csv_path": (
            "data/google-gemma-3-12b-pt-feature-act-small/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        ),
    },
    {
        "name": "Gemma-3-12B-IT-big",
        "csv_path": (
            "data/google-gemma-3-12b-it-feature-act-big/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        ),
    },
    {
        "name": "Gemma-3-12B-IT-small",
        "csv_path": (
            "data/google-gemma-3-12b-it-feature-act-small/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        ),
    },
    {
        "name": "Llama-3.1-8B",
        "csv_path": (
            "data/meta-llama-Llama-3.1-8B-feature-act/"
            "per_layer_per_emotion_feature_ablation/"
            "per_layer_per_emotion_ablation_summary.csv"
        ),
    },
]

OUT_DIR = "paper_visuals/all_models_layerwise_ablation"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_PATH_PNG = os.path.join(
    OUT_DIR,
    "all_models_layerwise_prediction_flip_rate_per_emotion.png",
)

OUT_PATH_PDF = os.path.join(
    OUT_DIR,
    "all_models_layerwise_prediction_flip_rate_per_emotion.pdf",
)


# ============================================================
# Style
# ============================================================

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

palette = sns.color_palette("tab10", n_colors=len(EMOTIONS))
emotion_to_color = dict(zip(EMOTIONS, palette))


# ============================================================
# Load data
# ============================================================

plot_dfs = []
valid_runs = []

for run in MODEL_RUNS:
    name = run["name"]
    csv_path = run["csv_path"]

    if not os.path.exists(csv_path):
        print(f"[Missing] {name}: {csv_path}")
        continue

    df = pd.read_csv(csv_path)

    required_cols = [
        "target_emotion",
        "layer_idx",
        "prediction_flip_rate",
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"[Skipping] {name}: missing columns {missing}")
        continue

    sub = df[required_cols].copy()

    sub["target_emotion"] = (
        sub["target_emotion"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    sub = sub[sub["target_emotion"].isin(EMOTIONS)].copy()

    sub["layer_idx"] = pd.to_numeric(
        sub["layer_idx"],
        errors="coerce",
    )

    sub["prediction_flip_rate"] = pd.to_numeric(
        sub["prediction_flip_rate"],
        errors="coerce",
    )

    sub = sub.dropna(
        subset=[
            "target_emotion",
            "layer_idx",
            "prediction_flip_rate",
        ]
    ).copy()

    sub["model"] = name

    plot_dfs.append(sub)
    valid_runs.append(run)

    print(f"Loaded {name}: {len(sub)} rows")

if len(plot_dfs) == 0:
    raise FileNotFoundError("No valid ablation summary files were found.")

all_df = pd.concat(plot_dfs, ignore_index=True)


# ============================================================
# Plot all models as subplots
# ============================================================

n_models = len(valid_runs)
ncols = 4
nrows = int(np.ceil(n_models / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(15.5, 9.8),
    sharey=True,
)

axes = np.array(axes).reshape(-1)

for ax, run in zip(axes, valid_runs):
    model_name = run["name"]
    model_df = all_df[all_df["model"] == model_name].copy()

    sns.lineplot(
        data=model_df,
        x="layer_idx",
        y="prediction_flip_rate",
        hue="target_emotion",
        hue_order=EMOTIONS,
        palette=emotion_to_color,
        marker="o",
        markersize=3.5,
        linewidth=1.6,
        ax=ax,
        legend=False,
    )

    ax.set_title(model_name, fontsize=9.5, fontweight="medium", pad=4)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Flip-away rate")
    ax.set_ylim(0, 1.0)
    ax.grid(True, linewidth=0.4, alpha=0.35)

    ax.tick_params(axis="both", length=2)

# Hide unused axes
for idx in range(n_models, len(axes)):
    axes[idx].axis("off")

# Shared legend
handles = [
    plt.Line2D(
        [0],
        [0],
        color=emotion_to_color[e],
        marker="o",
        linewidth=1.8,
        markersize=4,
        label=e.capitalize(),
    )
    for e in EMOTIONS
]

fig.legend(
    handles=handles,
    labels=[e.capitalize() for e in EMOTIONS],
    loc="lower center",
    ncol=len(EMOTIONS),
    frameon=False,
    bbox_to_anchor=(0.5, 0.005),
)

fig.subplots_adjust(
    left=0.055,
    right=0.99,
    top=0.94,
    bottom=0.09,
    wspace=0.22,
    hspace=0.32,
)

plt.savefig(OUT_PATH_PNG, dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(OUT_PATH_PDF, bbox_inches="tight", facecolor="white")

print("Saved:", OUT_PATH_PNG)
print("Saved:", OUT_PATH_PDF)

plt.show()

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns


# ============================================================
# Config
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-1b-pt",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-1b-it",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

EMOTIONS = ["joy", "surprise", "disgust", "anger", "sadness", "fear"]

BASE_DATA_DIR = "data"
OUTPUT_DIR = "paper_visuals/all_models_activation_swap_layerwise"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FLIP_TARGET_COL = "flip_to_target_rate_clean_source_only"

OUT_PATH_PNG = os.path.join(
    OUTPUT_DIR,
    "all_models_layerwise_flip_to_target_rate_per_emotion.png",
)

OUT_PATH_PDF = os.path.join(
    OUTPUT_DIR,
    "all_models_layerwise_flip_to_target_rate_per_emotion.pdf",
)


# ============================================================
# Helpers
# ============================================================

def get_sae_sizes_for_model(model_name):
    if model_name.startswith("google/gemma-2"):
        return [None]

    if model_name.startswith("meta-llama/"):
        return [None]

    if model_name.startswith("google/gemma-3"):
        return ["small", "big"]

    return [None]


def get_model_slug(model_name):
    return model_name.replace("/", "-")


def get_model_label(model_name, sae_size):
    slug = get_model_slug(model_name)

    if sae_size is None:
        return slug

    return f"{slug}-{sae_size}"


def get_feature_dir(model_name, sae_size):
    slug = get_model_slug(model_name)

    if sae_size is None:
        return os.path.join(BASE_DATA_DIR, f"{slug}-feature-act")

    return os.path.join(BASE_DATA_DIR, f"{slug}-feature-act-{sae_size}")


def prettify_model_label(label):
    label = label.replace("google-", "")
    label = label.replace("meta-llama-", "")
    label = label.replace("gemma-2-", "Gemma-2-")
    label = label.replace("gemma-3-", "Gemma-3-")
    label = label.replace("-small", " small")
    label = label.replace("-big", " big")
    return label


def make_sorting_key(prettified_label):
    is_llama = 1 if "Llama" in prettified_label else 0
    match = re.search(r"(\d+(?:\.\d+)?)[Bb]", prettified_label)
    size = float(match.group(1)) if match else 0.0
    is_big_sae = 1 if "big" in prettified_label else 0
    is_it = 1 if "it" in prettified_label.lower() else 0
    return (is_llama, size, is_it, is_big_sae)


# ============================================================
# Load swap summaries
# ============================================================

all_dfs = []
run_log = []

for model_name in MODELS:
    for sae_size in get_sae_sizes_for_model(model_name):
        model_label = get_model_label(model_name, sae_size)
        feature_dir = get_feature_dir(model_name, sae_size)

        summary_path = os.path.join(
            feature_dir,
            "cross_emotion_layer_feature_swap",
            "cross_emotion_layer_feature_swap_summary.csv",
        )

        if not os.path.exists(summary_path):
            print("[Missing]", summary_path)
            run_log.append(
                {
                    "model_name": model_name,
                    "sae_size": sae_size if sae_size is not None else "none",
                    "model_label": model_label,
                    "status": "missing",
                    "summary_path": summary_path,
                }
            )
            continue

        df_one = pd.read_csv(summary_path)

        required_cols = [
            "source_emotion",
            "target_emotion",
            "layer_idx",
            FLIP_TARGET_COL,
        ]

        missing = [c for c in required_cols if c not in df_one.columns]
        if missing:
            print("[Skipping]", summary_path, "missing:", missing)
            continue

        df_one["model_name"] = model_name
        df_one["sae_size"] = sae_size if sae_size is not None else "none"
        df_one["model_label"] = model_label
        df_one["model_display"] = prettify_model_label(model_label)

        all_dfs.append(df_one)

        print("[Loaded]", summary_path)

        run_log.append(
            {
                "model_name": model_name,
                "sae_size": sae_size if sae_size is not None else "none",
                "model_label": model_label,
                "status": "loaded",
                "summary_path": summary_path,
            }
        )

if len(all_dfs) == 0:
    raise FileNotFoundError("No activation swap summary files found.")

df = pd.concat(all_dfs, ignore_index=True)

df["source_emotion"] = df["source_emotion"].astype(str).str.lower().str.strip()
df["target_emotion"] = df["target_emotion"].astype(str).str.lower().str.strip()
df["layer_idx"] = pd.to_numeric(df["layer_idx"], errors="coerce")
df[FLIP_TARGET_COL] = pd.to_numeric(df[FLIP_TARGET_COL], errors="coerce")

df = df.dropna(subset=["layer_idx", FLIP_TARGET_COL]).copy()
df["layer_idx"] = df["layer_idx"].astype(int)

df = df[
    df["target_emotion"].isin(EMOTIONS)
    & df["source_emotion"].isin(EMOTIONS)
    & (df["source_emotion"] != df["target_emotion"])
].copy()


# ============================================================
# Aggregate: layer x target emotion per model
# ============================================================

plot_df = (
    df.groupby(
        ["model_display", "layer_idx", "target_emotion"],
        as_index=False,
    )
    .agg(
        flip_to_target_rate=(FLIP_TARGET_COL, "mean"),
    )
)

model_order = sorted(plot_df["model_display"].unique(), key=make_sorting_key)


# ============================================================
# Plot
# ============================================================

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

palette = sns.color_palette("tab10", n_colors=len(EMOTIONS))
emotion_to_color = dict(zip(EMOTIONS, palette))

n_models = len(model_order)
ncols = 4
nrows = int(np.ceil(n_models / ncols))

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(15.5, 10.0),
    sharey=True,
)

axes = np.array(axes).reshape(-1)

for ax, model_display in zip(axes, model_order):
    model_df = plot_df[plot_df["model_display"] == model_display].copy()

    sns.lineplot(
        data=model_df,
        x="layer_idx",
        y="flip_to_target_rate",
        hue="target_emotion",
        hue_order=EMOTIONS,
        palette=emotion_to_color,
        marker="o",
        markersize=3.5,
        linewidth=1.6,
        ax=ax,
        legend=False,
    )

    ax.set_title(model_display, fontsize=9.5, fontweight="medium", pad=4)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Flip-to-target rate")
    ax.set_ylim(0, 1.0)
    ax.grid(True, linewidth=0.4, alpha=0.35)
    ax.tick_params(axis="both", length=2)

for idx in range(n_models, len(axes)):
    axes[idx].axis("off")

handles = [
    plt.Line2D(
        [0],
        [0],
        color=emotion_to_color[e],
        marker="o",
        linewidth=1.8,
        markersize=4,
        label=e.capitalize(),
    )
    for e in EMOTIONS
]

fig.legend(
    handles=handles,
    labels=[e.capitalize() for e in EMOTIONS],
    loc="lower center",
    ncol=len(EMOTIONS),
    frameon=False,
    bbox_to_anchor=(0.5, 0.005),
)

fig.subplots_adjust(
    left=0.055,
    right=0.99,
    top=0.94,
    bottom=0.09,
    wspace=0.22,
    hspace=0.32,
)

plt.savefig(OUT_PATH_PNG, dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(OUT_PATH_PDF, bbox_inches="tight", facecolor="white")

print("Saved:", OUT_PATH_PNG)
print("Saved:", OUT_PATH_PDF)

plt.show()


# ============================================================
# Save source data
# ============================================================

plot_df.to_csv(
    os.path.join(OUTPUT_DIR, "all_models_layerwise_flip_to_target_rate_per_emotion.csv"),
    index=False,
)

pd.DataFrame(run_log).to_csv(
    os.path.join(OUTPUT_DIR, "run_log.csv"),
    index=False,
)